# DPD Role-Aware SAAMR: All-Atom PE/PEAA Ionomer to OpenMM

**Author:** Joseph R. Laforet Jr.

This notebook adapts the role-aware all-atom PE/PEAA ionomer workflow to use a notebook-local `AllAtomDPDPlacement` routine for polymer coordinate generation. DPD is used only as an initialization method for polymer atoms: the polymer chains are converted to one HOOMD particle per atom, bonded terms are labeled from an OpenFF `.offxml`, DPD pair forces are scaled from assigned vdW epsilon values, and final atom positions are written directly back to the MuPT hierarchy. Explicit Na+ counterions are placed afterward from the final PEAA carboxylate geometry and are not included in the DPD simulation.

The notebook intentionally keeps the all-atom DPD adapter notebook-local so we can prove the workflow before modifying MuPT source. Downstream SDF export, OpenFF parameterization, OpenMM minimization, and short MD smoke tests follow the original role-aware ionomer notebook.


## DPD Dependency Note

This notebook imports `hoomd`, `gsd`, and `freud` directly for notebook-local all-atom DPD placement. Use the notebook-specific environment in `conda-envs/dpd-ionomer-notebook-env.yml`. For this WSL2 workstation, the environment intentionally uses CPU HOOMD because GPU HOOMD pair forces fail at runtime under the available WSL2 CUDA managed-memory capabilities.


## HOOMD GPU Diagnosis Record

We attempted to run this DPD workflow on the RTX 3080 Ti Laptop GPU exposed through WSL2. The GPU is visible to Linux (`nvidia-smi`) and HOOMD can discover it with `hoomd.device.GPU.get_available_devices()`. However, minimal two-particle HOOMD GPU simulations with either `hoomd.md.pair.LJ` or `hoomd.md.pair.DPD` fail when the pair force attaches.

Diagnostic steps performed:

1. Created `mupt-dpd-ionomer` with conda-forge `hoomd=5.3.0` GPU build and verified imports for `hoomd`, `gsd`, `freud`, OpenFF, OpenMM, and PyTorch.
2. Confirmed `nvidia-smi` sees GPU 0 and HOOMD reports the device as available.
3. Ran minimal HOOMD GPU cases: no integrator, integrator only, and constant-volume no-force simulations succeed; adding GPU pair forces fails.
4. Reproduced the same failure in the pre-existing `hoomd-analysis` environment.
5. Tested a temporary `mupt-dpd-ionomer-gpu-hoomd4` environment with `hoomd=4.9.1` GPU build; minimal GPU pair forces still failed.
6. Queried CUDA runtime attributes. WSL2 reports `managedMemory=1` but `concurrentManagedAccess=0` and `pageableMemoryAccess=0`. HOOMD 5.3.1 reports this more directly as: `The device NVIDIA GeForce RTX 3080 Ti Laptop GPU does not support managed memory.`

Observed failure signature for GPU pair forces:

```text
RuntimeError: CUDA Error: invalid device ordinal before /hoomd/Autotuner.h:497
```

Conclusion: the misleading `invalid device ordinal` is triggered by HOOMD GPU pair-force/autotuner execution under the current WSL2 CUDA capability report, not by a missing GPU or wrong GPU index. Until this is tested on native Linux or a different HOOMD/CUDA runtime, this notebook forces CPU execution.


In [23]:
from __future__ import annotations

from dataclasses import dataclass
from collections import Counter, defaultdict
from pathlib import Path
import sys
import os
import json
import math
import time
import logging
import shutil
import subprocess

import networkx as nx
import numpy as np
from anytree import PreOrderIter
from rdkit import Chem
from rdkit.Geometry import Point3D
from scipy.spatial import cKDTree
from scipy.spatial.transform import RigidTransform, Rotation
try:
    from tqdm.auto import tqdm
except ModuleNotFoundError:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else range(0)


def find_examples_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for candidate in (start, *start.parents):
        if (candidate / "examples_system").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not locate the mupt-examples repository root")


EXAMPLES_ROOT = find_examples_root()
LOCAL_MUPT_SOURCE = EXAMPLES_ROOT / "mupt"
# Do not hide CUDA globally; OpenMM should still be able to use the GPU.
# HOOMD is forced onto CPU explicitly when its Simulation is constructed.
if LOCAL_MUPT_SOURCE.exists():
    sys.path.insert(0, str(LOCAL_MUPT_SOURCE))

try:
    import freud
    import gsd.hoomd
    import hoomd
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "This DPD notebook requires hoomd, gsd, and freud. "
        "Install/use the MuPT MD environment before running it."
    ) from exc
from mupt.geometry.coordinates.reference import origin
from mupt.geometry.shapes import Ellipsoid, PointCloud
from mupt.geometry.transforms.rigid import rigid_vector_coalignment
from mupt.interfaces.rdkit import primitive_to_rdkit_mols
from mupt.interfaces._shared.topology import build_saamr_role_topology_index, _resolve_to_atom
from mupt.interfaces.smiles import primitive_from_smiles
from mupt.mupr.primitives import Primitive
from mupt.mupr.topology import TopologicalStructure
from mupt.roles import PrimitiveRole

logging.getLogger("mupt.mupr.primitives").setLevel(logging.ERROR)

def resolve_to_atom_cached(parent: Primitive, conn_ref, cache: dict[tuple[int, object, object], Primitive]) -> Primitive:
    cache_key = (id(parent), conn_ref.primitive_handle, conn_ref.connector_handle)
    if cache_key not in cache:
        cache[cache_key] = _resolve_to_atom(parent, conn_ref)
    return cache[cache_key]

OUTPUT_ROOT = EXAMPLES_ROOT / "examples_system" / "dpd_role_aware_ionomer_outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {EXAMPLES_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")

Repository root: /home/joelaforet/Shirts-Lab-Linux/mupt-examples
Output root: /home/joelaforet/Shirts-Lab-Linux/mupt-examples/examples_system/dpd_role_aware_ionomer_outputs


## 1. Science Knobs

`USE_PRODUCTION_SIZE = False` keeps the notebook runnable as a demonstration. Set it to `True` to build 800 chains for the selected system.

In [24]:
SYSTEM_SPECS = {
    "m3": {
        "pattern": ["PE", "PEAA", "PE"],
        "repeat_count": 12,
        "expected_units": 36,
    },
    "m5": {
        "pattern": ["PE", "PE", "PEAA", "PE", "PE"],
        "repeat_count": 7,
        "expected_units": 35,
    },
    "m7": {
        "pattern": ["PE", "PE", "PE", "PEAA", "PE", "PE", "PE"],
        "repeat_count": 5,
        "expected_units": 35,
    },
}

BUILD_SYSTEM_NAME = "m5"  # "m3", "m5", "m7", or "all"
USE_PRODUCTION_SIZE = True
N_CHAINS_SMOKE_TEST = 2
N_CHAINS_PRODUCTION = 200
RANDOM_SEED = 51

DA_PER_NM3_TO_G_CM3 = 1.0 / 602.214076
TARGET_INITIAL_DENSITY_G_CM3 = 0.80
INTER_RESIDUE_BOND_LENGTH_A = 2.8
SODIUM_COO_DISTANCE_A = 2.4
SODIUM_COO_DISTANCE_CANDIDATES_A = (2.4, 3.0, 3.6, 4.2)
SODIUM_MIN_POLYMER_DISTANCE_A = 2.0
SODIUM_MIN_SODIUM_DISTANCE_A = 2.5
SODIUM_PLACEMENT_CANDIDATES = 512

# All-atom DPD placement parameters. Coordinates are in Angstrom and energies
# follow the OpenFF bonded/nonbonded parameter units (kcal/mol). DPD pair A and
# gamma are scaled by assigned OpenFF vdW epsilon values, matching the approach
# in examples_DPD/DPD_AA.ipynb.
DPD_OPENFF_FORCE_FIELD = "openff-2.2.1.offxml"
DPD_ATOM_NUMBER_DENSITY_A3 = 0.025  # legacy fallback when target density is disabled
DPD_R_CUT_A = 3.5
DPD_KT_KCAL_MOL = 1.0
DPD_A_BASE = 5_000.0
DPD_GAMMA_BASE = 800.0
DPD_DT = 0.001
DPD_DEVICE = "GPU"  # "GPU", "CPU", or "auto"
DPD_NLIST_EXCLUSIONS = ("bond", "angle", "dihedral")
DPD_INITIALIZATION_MODE = "repeat_unit_random_walk"  # "repeat_unit_random_walk" or "chains"
DPD_PARTICLE_SPACING_A = 0.75
DPD_INITIAL_CHAIN_STEP_A = 5.0
DPD_REPEAT_WALK_STEP_A = INTER_RESIDUE_BOND_LENGTH_A
DPD_STEPS_PER_INTERVAL = 1_000
DPD_STEPS_MAX = None  # None means run until geometry converges
DPD_REPORT_INTERVAL = 1_000
DPD_BOND_K_SCALE = 1.0
DPD_ANGLE_K_SCALE = 1.0
DPD_DIHEDRAL_K_SCALE = 1.0
DPD_EPSILON_REFERENCE_MODE = "max"  # "max", "mean", or a positive float
WRITE_DPD_GSD = False
WRITE_DPD_PYMOL_MOVIE = True
DPD_PYMOL_FRAME_INTERVAL_STEPS = 1_000
DPD_PYMOL_MAX_FRAMES = 120
MIN_ALLOWED_DISTANCE_A = 0.75
DPD_MAX_BOND_DISTANCE_A = 3.0
DPD_REQUIRE_GEOMETRY_CONVERGENCE = True
DPD_LONG_BOND_DETAIL_LIMIT = 12
DPD_CLOSE_CONTACT_DETAIL_LIMIT = 12
DPD_PLATEAU_WINDOW_INTERVALS = 20
DPD_PLATEAU_MIN_BOND_IMPROVEMENT = 1
DIAGNOSTIC_MODE = True

N_CHAINS = N_CHAINS_PRODUCTION if USE_PRODUCTION_SIZE else N_CHAINS_SMOKE_TEST
SELECTED_SYSTEMS = list(SYSTEM_SPECS) if BUILD_SYSTEM_NAME == "all" else [BUILD_SYSTEM_NAME]

print(f"Selected systems: {SELECTED_SYSTEMS}")
print(f"Chains per system: {N_CHAINS}")
print(f"Target initial density: {TARGET_INITIAL_DENSITY_G_CM3:.3f} g/cm^3")

Selected systems: ['m5']
Chains per system: 200


## 2. Repeat Chemistry

Each `PE` or `PEAA` repeat represents a three-carbon backbone block. The acid-bearing repeat is `-CH2-CH(CH2COO-)-CH2-`. Head and tail cap residues have one linker site and explicit terminal hydrogen caps; they do not add extra carbons. Sodium is represented as a separate one-particle residue and is not covalently bonded to the polymer.

In [25]:
REPEAT_SMILES = {
    "PE_HEAD": "[H]-[CH2:1]-[CH2]-[CH2:2]-*",
    "PE": "*-[CH2:1]-[CH2]-[CH2:2]-*",
    "PEAA": "*-[CH2:1]-[CH]([CH2][C](=O)[O-])-[CH2:2]-*",
    "PE_TAIL": "*-[CH2:1]-[CH2]-[CH2:2]-[H]",
    "NA": "[Na+]",
}

# Capped analogues are used only to obtain local conformers for the linker-bearing
# repeat templates. The MuPT topology is still built from REPEAT_SMILES above.
REPEAT_CONFORMER_SMILES = {
    "PE_HEAD": "[H]-[CH2:1]-[CH2]-[CH2:2]-[H]",
    "PE": "[H]-[CH2:1]-[CH2]-[CH2:2]-[H]",
    "PEAA": "[H]-[CH2:1]-[CH]([CH2][C](=O)[O-])-[CH2:2]-[H]",
    "PE_TAIL": "[H]-[CH2:1]-[CH2]-[CH2:2]-[H]",
}

RESNAME_MAP = {
    "PE_HEAD": "PEH",
    "PE": "PEX",
    "PEAA": "PAA",
    "PE_TAIL": "PET",
    "NA": "SOD",
}

PEAA_CARBOXYLATE_CARBON_ATOM_LABEL = 4
PEAA_CARBOXYLATE_OXYGEN_ATOM_LABELS = (5, 6)
AXIS = 0
SEMIMINOR_FRACT = 0.5


## 3. Notebook-Local Builder

This is intentionally notebook-local. The point is to demonstrate that MuPT can already represent the role-aware hierarchy, cap residues, counterions, SDF export, and OpenMM handoff without waiting for a specialized public ionomer builder API.

In [26]:
@dataclass(frozen=True)
class BuiltIonomerSystem:
    primitive: Primitive
    system_name: str
    chain_sequences: list[list[str]]
    sodium_count: int
    sdf_paths: list[Path]


def template_positions_from_smiles(smiles: str, random_seed: int = 52) -> dict[int, np.ndarray]:
    """Return RDKit conformer positions keyed by RDKit atom index."""
    template = primitive_from_smiles(
        smiles,
        label="template",
        ensure_explicit_Hs=True,
        embed_positions=True,
    )
    return {
        int(atom.label): np.array(atom.shape.centroid, dtype=float)
        for atom in template.children
    }


def assign_repeat_template_geometry(residue: Primitive, conformer_smiles: str) -> None:
    """Assign RDKit-generated local coordinates and an envelope shape to a repeat."""
    positions_by_label = template_positions_from_smiles(conformer_smiles)
    points = []
    for atom in residue.children:
        position = positions_by_label[int(atom.label)]
        atom.shape = PointCloud(position)
        points.append(position)
    residue.shape = PointCloud(np.vstack(points))

    head_atom, tail_atom = residue.search_hierarchy_by(
        lambda prim: "molAtomMapNumber" in prim.metadata,
        min_count=2,
    )
    head_pos = np.array(head_atom.shape.centroid, dtype=float)
    tail_pos = np.array(tail_atom.shape.centroid, dtype=float)
    major_radius = np.linalg.norm(tail_pos - head_pos) / 2.0
    axis_vec = np.zeros(3, dtype=float)
    axis_vec[AXIS] = major_radius
    residue.rigidly_transform(
        rigid_vector_coalignment(
            vector1_start=head_pos,
            vector1_end=tail_pos,
            vector2_start=origin(3),
            vector2_end=axis_vec,
            t1=0.5,
            t2=0.0,
        )
    )

    semiminor = SEMIMINOR_FRACT * major_radius
    radii = np.full(3, semiminor)
    radii[AXIS] = major_radius
    residue.shape = Ellipsoid(radii)

    for conn_handle, conn_ref in residue.external_connectors.items():
        atom = residue.children_by_handle[conn_ref.primitive_handle]
        anchor = np.array(atom.shape.centroid, dtype=float)
        direction = -1.0 if atom.metadata.get("molAtomMapNumber") == 1 else 1.0
        linker = anchor + np.array([direction * INTER_RESIDUE_BOND_LENGTH_A, 0.0, 0.0])
        for conn in (residue.fetch_connector(conn_handle), residue.fetch_connector_on_child(conn_ref)):
            conn.anchor.position = anchor
            conn.linker.position = linker


def residue_atom_position(residue: Primitive, atom_label: int) -> np.ndarray:
    """Return the centroid of the atom with the requested RDKit atom label."""
    for atom in residue.children:
        if int(atom.label) == atom_label:
            return np.array(atom.shape.centroid, dtype=float)
    raise KeyError(f"Could not find atom label {atom_label} in {residue.label}")


def set_single_atom_residue_position(residue: Primitive, position: np.ndarray) -> None:
    """Move a one-particle residue to an absolute position."""
    atom = residue.children[0]
    atom.shape = PointCloud(np.array(position, dtype=float))
    residue.shape = PointCloud(np.array(position, dtype=float))


def random_unit_vector(rng: np.random.Generator) -> np.ndarray:
    """Return a random unit vector."""
    vector = rng.normal(size=3)
    norm = np.linalg.norm(vector)
    if norm < 1.0e-12:
        return np.array([1.0, 0.0, 0.0])
    return vector / norm


def rotation_between_vectors(source: np.ndarray, target: np.ndarray) -> Rotation:
    """Return a rotation that maps source direction onto target direction."""
    source_norm = np.linalg.norm(source)
    target_norm = np.linalg.norm(target)
    if source_norm < 1.0e-12 or target_norm < 1.0e-12:
        return Rotation.identity()
    source_unit = np.asarray(source, dtype=float) / source_norm
    target_unit = np.asarray(target, dtype=float) / target_norm
    return Rotation.align_vectors([target_unit], [source_unit])[0]


def repeat_connector_atoms(residue: Primitive) -> tuple[Primitive, Primitive]:
    """Return head and tail backbone connector atoms for a repeat residue."""
    mapped_atoms = {}
    for atom in residue.children:
        map_number = atom.metadata.get("molAtomMapNumber")
        if map_number is not None:
            mapped_atoms[int(map_number)] = atom
    if 1 not in mapped_atoms or 2 not in mapped_atoms:
        raise ValueError(f"Residue {residue.label!r} is missing connector atoms with molAtomMapNumber 1 and 2")
    return mapped_atoms[1], mapped_atoms[2]


def transform_residue_atoms(residue: Primitive, rotation: Rotation, source_anchor: np.ndarray, target_anchor: np.ndarray) -> None:
    """Rigidly rotate residue atoms around source_anchor, then translate to target_anchor."""
    points = []
    for atom in residue.children:
        old_position = np.asarray(atom.shape.centroid, dtype=float)
        new_position = target_anchor + rotation.apply(old_position - source_anchor)
        atom.shape = PointCloud(new_position)
        points.append(new_position)
    residue.shape = PointCloud(np.vstack(points))


def wrapped_positions_from_primitives(primitives: list[Primitive], box_lengths: np.ndarray) -> np.ndarray:
    """Return primitive atom positions wrapped into an orthorhombic periodic box."""
    positions = [np.asarray(atom.shape.centroid, dtype=float) for atom in primitives]
    if not positions:
        return np.empty((0, 3), dtype=float)
    return np.asarray(positions, dtype=float) % box_lengths


def place_sodium_near_peaa(
    peaa_residue: Primitive,
    rng: np.random.Generator,
    polymer_tree: cKDTree | None = None,
    placed_sodium_positions: list[np.ndarray] | None = None,
    box_lengths: np.ndarray | None = None,
) -> np.ndarray:
    """Return a sodium position near a carboxylate while avoiding periodic clashes."""
    oxy_1 = residue_atom_position(peaa_residue, PEAA_CARBOXYLATE_OXYGEN_ATOM_LABELS[0])
    oxy_2 = residue_atom_position(peaa_residue, PEAA_CARBOXYLATE_OXYGEN_ATOM_LABELS[1])
    carbon = residue_atom_position(peaa_residue, PEAA_CARBOXYLATE_CARBON_ATOM_LABEL)
    midpoint = 0.5 * (oxy_1 + oxy_2)
    direction = midpoint - carbon
    if np.linalg.norm(direction) < 1.0e-8:
        direction = random_unit_vector(rng)
    direction = direction / np.linalg.norm(direction)

    if polymer_tree is None or box_lengths is None:
        jitter = 0.05 * rng.normal(size=3)
        direction = direction + jitter
        direction = direction / np.linalg.norm(direction)
        return midpoint + SODIUM_COO_DISTANCE_A * direction

    best_position = midpoint + SODIUM_COO_DISTANCE_A * direction
    best_score = -np.inf
    sodium_tree = None
    if placed_sodium_positions:
        sodium_tree = cKDTree(np.asarray(placed_sodium_positions, dtype=float), boxsize=box_lengths)

    directions = [direction]
    directions.extend(random_unit_vector(rng) for _ in range(SODIUM_PLACEMENT_CANDIDATES - 1))
    for radius_a in SODIUM_COO_DISTANCE_CANDIDATES_A:
        for candidate_direction in directions:
            candidate = midpoint + radius_a * candidate_direction
            wrapped_candidate = candidate % box_lengths
            polymer_distance = float(polymer_tree.query(wrapped_candidate, k=1)[0])
            sodium_distance = 999.0
            if sodium_tree is not None:
                sodium_distance = float(sodium_tree.query(wrapped_candidate, k=1)[0])
            score = min(polymer_distance, sodium_distance)
            if score > best_score:
                best_score = score
                best_position = candidate
            if polymer_distance >= SODIUM_MIN_POLYMER_DISTANCE_A and sodium_distance >= SODIUM_MIN_SODIUM_DISTANCE_A:
                return candidate

    return best_position


def build_repeat_lexicon() -> dict[str, Primitive]:
    """Create RESIDUE -> PARTICLE primitives for repeat units and sodium."""
    lexicon = {}
    for name, smiles in REPEAT_SMILES.items():
        residue = primitive_from_smiles(
            smiles,
            label=name,
            ensure_explicit_Hs=True,
            embed_positions=False,
        )
        residue.role = PrimitiveRole.RESIDUE
        residue.metadata.update({
            "repeat_kind": name,
            "residue_name": RESNAME_MAP[name],
        })
        for atom in residue.children:
            atom.role = PrimitiveRole.PARTICLE
            atom.metadata.setdefault("residue_name", RESNAME_MAP[name])
        if name != "NA":
            assign_repeat_template_geometry(residue, REPEAT_CONFORMER_SMILES[name])
        lexicon[name] = residue
    return lexicon


def expanded_sequence(system_name: str) -> list[str]:
    """Expand a system pattern and replace terminal PE blocks with cap residues."""
    spec = SYSTEM_SPECS[system_name]
    sequence = list(spec["pattern"]) * int(spec["repeat_count"])
    if len(sequence) != int(spec["expected_units"]):
        raise ValueError(f"{system_name}: expected {spec['expected_units']} units, got {len(sequence)}")
    if sequence[0] != "PE" or sequence[-1] != "PE":
        raise ValueError("Current cap logic expects chain patterns to start and end with PE")
    sequence[0] = "PE_HEAD"
    sequence[-1] = "PE_TAIL"
    return sequence


def random_rotation(rng: np.random.Generator):
    """Return a uniformly random 3D rotation."""
    return Rotation.random(random_state=rng)


def wrap_positions(positions: np.ndarray, box_lengths: np.ndarray) -> np.ndarray:
    """Wrap coordinates into a centered orthorhombic box."""
    return ((positions + 0.5 * box_lengths) % box_lengths) - 0.5 * box_lengths


def primitive_mass_da(root: Primitive) -> float:
    """Return total mass of atomic Primitive leaves in daltons."""
    mass = 0.0
    for atom in root.leaves:
        if atom.element is None:
            raise ValueError(f"Atomic Primitive '{atom.label}' has no element for density-first box sizing")
        mass += float(atom.element.mass)
    return mass


def density_box_length_a(mass_da: float, density_g_cm3: float) -> float:
    """Return cubic box length in Angstrom for a target mass density."""
    if density_g_cm3 <= 0.0:
        raise ValueError("density_g_cm3 must be positive")
    volume_nm3 = mass_da * DA_PER_NM3_TO_G_CM3 / density_g_cm3
    return 10.0 * float(np.cbrt(volume_nm3))


def cubic_density_g_cm3(mass_da: float, box_length_a: float) -> float:
    """Return mass density for a cubic box length in Angstrom."""
    volume_nm3 = (box_length_a * 0.1) ** 3
    return mass_da * DA_PER_NM3_TO_G_CM3 / volume_nm3


def make_hoomd_device():
    """Create requested HOOMD device, falling back to CPU if GPU is unavailable."""
    requested = DPD_DEVICE.lower()
    if requested == "cpu":
        return hoomd.device.CPU()
    if requested == "gpu":
        try:
            return hoomd.device.GPU()
        except Exception as exc:
            print(f"HOOMD GPU unavailable ({exc}); falling back to CPU")
            return hoomd.device.CPU()
    if requested == "auto":
        return hoomd.device.auto_select()
    raise ValueError(f"Unknown DPD_DEVICE={DPD_DEVICE!r}; expected GPU, CPU, or auto")


class AllAtomDPDPlacement:
    """Notebook-local all-atom DPD placement using OpenFF-labeled parameters."""

    def __init__(self, random_seed: int) -> None:
        self.rng = np.random.default_rng(random_seed)
        self.summary = {}

    def _box_length(
        self,
        n_atoms: int,
        total_mass_da: float | None = None,
        target_density_g_cm3: float | None = None,
    ) -> float:
        if target_density_g_cm3 is not None:
            if total_mass_da is None:
                raise ValueError("total_mass_da is required for density-first DPD box sizing")
            length = density_box_length_a(total_mass_da, target_density_g_cm3)
        else:
            length = float(np.cbrt(n_atoms / DPD_ATOM_NUMBER_DENSITY_A3))
        return max(length, 3.0 * DPD_R_CUT_A)

    def _initialize_polymer_positions(self, polymer_chain_residues: list[list[Primitive]], box_length: float) -> None:
        """Place intact template residues as randomly oriented chains before DPD."""
        for residues in polymer_chain_residues:
            chain_center = self.rng.uniform(-0.5 * box_length, 0.5 * box_length, size=3)
            rotation = random_rotation(self.rng)
            center_offset = 0.5 * (len(residues) - 1) * DPD_INITIAL_CHAIN_STEP_A
            for residue_idx, residue in enumerate(residues):
                old_center = np.asarray(residue.shape.centroid, dtype=float)
                new_center = np.array([residue_idx * DPD_INITIAL_CHAIN_STEP_A - center_offset, 0.0, 0.0])
                new_center = chain_center + rotation.apply(new_center)
                points = []
                for atom in residue.children:
                    local_position = np.asarray(atom.shape.centroid, dtype=float) - old_center
                    new_position = new_center + rotation.apply(local_position)
                    atom.shape = PointCloud(new_position)
                    points.append(new_position)
                residue.shape = PointCloud(np.vstack(points))

    def _initialize_repeat_unit_random_walk(self, polymer_chain_residues: list[list[Primitive]], box_length: float) -> None:
        """Place repeat units by a connector-aware non-self-avoiding random walk."""
        for chain_idx, residues in enumerate(polymer_chain_residues):
            previous_tail_position = None
            direction = random_unit_vector(self.rng)
            first_head_position = self.rng.uniform(-0.5 * box_length, 0.5 * box_length, size=3)
            for residue_idx, residue in enumerate(residues):
                head_atom, tail_atom = repeat_connector_atoms(residue)
                head_position = np.asarray(head_atom.shape.centroid, dtype=float)
                tail_position = np.asarray(tail_atom.shape.centroid, dtype=float)
                current_axis = tail_position - head_position
                if residue_idx == 0:
                    target_head_position = first_head_position
                else:
                    direction = random_unit_vector(self.rng)
                    target_head_position = previous_tail_position + DPD_REPEAT_WALK_STEP_A * direction
                rotation = rotation_between_vectors(current_axis, direction)
                transform_residue_atoms(residue, rotation, head_position, target_head_position)
                previous_tail_position = np.asarray(tail_atom.shape.centroid, dtype=float)
            if DIAGNOSTIC_MODE and (chain_idx + 1) % 25 == 0:
                print(f"DPD diagnostic: connector-aware repeat-unit random-walk initialized {chain_idx + 1}/{len(polymer_chain_residues)} chains")

    def _segment_records(self, polymer_root: Primitive) -> list[dict]:
        index = build_saamr_role_topology_index(polymer_root)
        records = []
        endpoint_cache = {}
        global_atom_idx = 0
        for segment in index.segments:
            atoms = []
            for residue in index.residues_by_segment[id(segment)]:
                atoms.extend(index.particles_by_residue[id(residue)])
            atom_to_local = {id(atom): idx for idx, atom in enumerate(atoms)}
            local_to_global = list(range(global_atom_idx, global_atom_idx + len(atoms)))
            global_atom_idx += len(atoms)
            bonds = []
            for node in index.bond_nodes:
                if index.segment_of_node[id(node)] is not segment:
                    continue
                for conn_pair in node.internal_connections:
                    conn_refs = tuple(sorted(conn_pair, key=lambda ref: (repr(ref.primitive_handle), repr(ref.connector_handle))))
                    atom_1 = resolve_to_atom_cached(node, conn_refs[0], endpoint_cache)
                    atom_2 = resolve_to_atom_cached(node, conn_refs[1], endpoint_cache)
                    bonds.append(tuple(sorted((atom_to_local[id(atom_1)], atom_to_local[id(atom_2)]))))
            records.append({"segment": segment, "atoms": atoms, "local_to_global": local_to_global, "bonds": sorted(set(bonds))})
        return records

    def _openff_labels(self, polymer_root: Primitive, resname_map: dict[str, str]):
        from openff.toolkit import ForceField, Molecule, Topology
        from openff.units import unit as off_unit

        rdkit_mol = next(iter(primitive_to_rdkit_mols(polymer_root, resname_map=resname_map, default_atom_position=np.zeros(3))))
        off_mol = Molecule.from_rdkit(rdkit_mol, allow_undefined_stereo=True, hydrogens_are_explicit=True)
        force_field = ForceField(DPD_OPENFF_FORCE_FIELD)
        labels = force_field.label_molecules(Topology.from_molecules([off_mol]))[0]
        return labels, off_unit

    def _epsilon_reference(self, epsilon_by_type: dict[str, float]) -> float:
        mode = DPD_EPSILON_REFERENCE_MODE
        if isinstance(mode, (int, float)):
            return float(mode)
        values = np.array(list(epsilon_by_type.values()), dtype=float)
        return float(values.mean() if mode == "mean" else values.max())

    def _parameter_tables(self, records: list[dict], labels, off_unit):
        first_atoms = records[0]["atoms"]
        atom_types_by_local = {}
        epsilon_by_type = {}
        for (local_idx,), parameter in labels["vdW"].items():
            element = first_atoms[local_idx].element.symbol
            atom_type = f"{element}_{parameter.id}"
            atom_types_by_local[local_idx] = atom_type
            epsilon_by_type[atom_type] = float(parameter.epsilon.m_as(off_unit.kilocalorie_per_mole))

        epsilon_reference = self._epsilon_reference(epsilon_by_type)
        particle_types = sorted(epsilon_by_type)
        pair_params = {}
        for i, type_i in enumerate(particle_types):
            for type_j in particle_types[i:]:
                epsilon_ij = math.sqrt(epsilon_by_type[type_i] * epsilon_by_type[type_j])
                scale = epsilon_ij / epsilon_reference if epsilon_reference > 0 else 1.0
                pair_params[(type_i, type_j)] = {"A": DPD_A_BASE * scale, "gamma": DPD_GAMMA_BASE * scale}

        bond_params = {}
        bond_type_by_local = {}
        for local_pair, parameter in labels["Bonds"].items():
            i, j = tuple(local_pair)
            bond_type = f"{atom_types_by_local[i]}-{atom_types_by_local[j]}_{parameter.id}"
            bond_type_by_local[tuple(sorted((i, j)))] = bond_type
            bond_params[bond_type] = {
                "k": float(parameter.k.m_as(off_unit.kilocalorie_per_mole / off_unit.angstrom**2)) * DPD_BOND_K_SCALE,
                "r0": float(parameter.length.m_as(off_unit.angstrom)),
            }

        angle_params = {}
        angle_type_by_local = {}
        for local_triplet, parameter in labels["Angles"].items():
            i, j, k = tuple(local_triplet)
            angle_type = f"{atom_types_by_local[i]}-{atom_types_by_local[j]}-{atom_types_by_local[k]}_{parameter.id}"
            angle_type_by_local[(i, j, k)] = angle_type
            angle_params[angle_type] = {
                "k": float(parameter.k.m_as(off_unit.kilocalorie_per_mole / off_unit.radian**2)) * DPD_ANGLE_K_SCALE,
                "t0": float(parameter.angle.m_as(off_unit.radian)),
            }

        dihedral_params = {}
        dihedral_terms_by_local = defaultdict(list)
        for local_quad, parameter in labels["ProperTorsions"].items():
            i, j, k, l = tuple(local_quad)
            for term_idx, k_term in enumerate(parameter.k):
                dihedral_type = f"{atom_types_by_local[i]}-{atom_types_by_local[j]}-{atom_types_by_local[k]}-{atom_types_by_local[l]}_{parameter.id}_{term_idx}"
                idivf = float(parameter.idivf[term_idx]) if hasattr(parameter, "idivf") else 1.0
                k_value = float(k_term.m_as(off_unit.kilocalorie_per_mole)) * DPD_DIHEDRAL_K_SCALE / idivf
                dihedral_params[dihedral_type] = {
                    "k": abs(k_value),
                    "d": 1 if k_value >= 0 else -1,
                    "n": int(parameter.periodicity[term_idx]),
                    "phi0": float(parameter.phase[term_idx].m_as(off_unit.radian)),
                }
                dihedral_terms_by_local[(i, j, k, l)].append(dihedral_type)

        return {
            "atom_types_by_local": atom_types_by_local,
            "particle_types": particle_types,
            "pair_params": pair_params,
            "bond_type_by_local": bond_type_by_local,
            "bond_params": bond_params,
            "angle_type_by_local": angle_type_by_local,
            "angle_params": angle_params,
            "dihedral_terms_by_local": dihedral_terms_by_local,
            "dihedral_params": dihedral_params,
        }

    def _build_frame(self, records: list[dict], tables: dict, box_length: float):
        atom_count = sum(len(record["atoms"]) for record in records)
        frame = gsd.hoomd.Frame()
        frame.particles.N = atom_count
        frame.particles.types = tables["particle_types"]
        type_to_id = {particle_type: idx for idx, particle_type in enumerate(frame.particles.types)}
        positions = np.zeros((atom_count, 3), dtype=float)
        typeids = np.zeros(atom_count, dtype=np.int32)
        bonds = []
        bond_typeids = []
        angles = []
        angle_typeids = []
        dihedrals = []
        dihedral_typeids = []
        bond_types = sorted(tables["bond_params"])
        angle_types = sorted(tables["angle_params"])
        dihedral_types = sorted(tables["dihedral_params"])
        bond_type_to_id = {name: idx for idx, name in enumerate(bond_types)}
        angle_type_to_id = {name: idx for idx, name in enumerate(angle_types)}
        dihedral_type_to_id = {name: idx for idx, name in enumerate(dihedral_types)}

        chain_atom_indices = []
        bond_adjacency = defaultdict(set)
        for record in records:
            chain_atom_indices.append(record["local_to_global"])
            for local_idx, atom in enumerate(record["atoms"]):
                global_idx = record["local_to_global"][local_idx]
                positions[global_idx] = np.asarray(atom.shape.centroid, dtype=float)
                typeids[global_idx] = type_to_id[tables["atom_types_by_local"][local_idx]]
            for i, j in record["bonds"]:
                global_i = record["local_to_global"][i]
                global_j = record["local_to_global"][j]
                bond_type = tables["bond_type_by_local"][tuple(sorted((i, j)))]
                bonds.append((global_i, global_j))
                bond_typeids.append(bond_type_to_id[bond_type])
                bond_adjacency[global_i].add(global_j)
                bond_adjacency[global_j].add(global_i)

            local_neighbors = defaultdict(set)
            for i, j in record["bonds"]:
                local_neighbors[i].add(j)
                local_neighbors[j].add(i)
            for j, neighbors in local_neighbors.items():
                sorted_neighbors = sorted(neighbors)
                for neighbor_idx, i in enumerate(sorted_neighbors):
                    for k in sorted_neighbors[neighbor_idx + 1:]:
                        angle_key = (i, j, k) if (i, j, k) in tables["angle_type_by_local"] else (k, j, i)
                        angle_type = tables["angle_type_by_local"].get(angle_key)
                        if angle_type is None:
                            continue
                        angles.append(tuple(record["local_to_global"][idx] for idx in angle_key))
                        angle_typeids.append(angle_type_to_id[angle_type])
            seen_dihedrals = set()
            for j, k in record["bonds"]:
                for i in local_neighbors[j] - {k}:
                    for l in local_neighbors[k] - {j}:
                        quad = (i, j, k, l)
                        reverse_quad = tuple(reversed(quad))
                        lookup_quad = quad if quad in tables["dihedral_terms_by_local"] else reverse_quad
                        canonical = min(quad, reverse_quad)
                        if canonical in seen_dihedrals or lookup_quad not in tables["dihedral_terms_by_local"]:
                            continue
                        seen_dihedrals.add(canonical)
                        for dihedral_type in tables["dihedral_terms_by_local"][lookup_quad]:
                            dihedrals.append(tuple(record["local_to_global"][idx] for idx in lookup_quad))
                            dihedral_typeids.append(dihedral_type_to_id[dihedral_type])

        box_lengths = np.array([box_length, box_length, box_length], dtype=float)
        frame.configuration.box = [box_length, box_length, box_length, 0, 0, 0]
        frame.particles.position = wrap_positions(positions, box_lengths)
        frame.particles.typeid = typeids
        frame.bonds.N = len(bonds)
        frame.bonds.types = bond_types
        frame.bonds.typeid = np.asarray(bond_typeids, dtype=np.int32)
        frame.bonds.group = np.asarray(bonds, dtype=np.int32)
        frame.angles.N = len(angles)
        frame.angles.types = angle_types
        frame.angles.typeid = np.asarray(angle_typeids, dtype=np.int32)
        frame.angles.group = np.asarray(angles, dtype=np.int32)
        frame.dihedrals.N = len(dihedrals)
        frame.dihedrals.types = dihedral_types
        frame.dihedrals.typeid = np.asarray(dihedral_typeids, dtype=np.int32)
        frame.dihedrals.group = np.asarray(dihedrals, dtype=np.int32)
        return frame, chain_atom_indices, bond_adjacency

    def _minimum_spacing_ok(self, snapshot) -> bool:
        query = freud.locality.AABBQuery(snapshot.configuration.box, snapshot.particles.position)
        neighbors = query.query(snapshot.particles.position, {"r_min": 0.0, "r_max": DPD_PARTICLE_SPACING_A, "exclude_ii": True}).toNeighborList()
        return len(neighbors) == 0

    def _excluded_pairs_from_frame(self, frame) -> set[tuple[int, int]]:
        """Return particle pairs excluded from DPD, matching DPD_NLIST_EXCLUSIONS."""
        excluded_pairs = set()
        if "bond" in DPD_NLIST_EXCLUSIONS:
            for atom_i, atom_j in np.asarray(frame.bonds.group, dtype=np.int32):
                excluded_pairs.add(tuple(sorted((int(atom_i), int(atom_j)))))
        if "angle" in DPD_NLIST_EXCLUSIONS:
            for atom_i, _, atom_k in np.asarray(frame.angles.group, dtype=np.int32):
                excluded_pairs.add(tuple(sorted((int(atom_i), int(atom_k)))))
        if "dihedral" in DPD_NLIST_EXCLUSIONS:
            for atom_i, _, _, atom_l in np.asarray(frame.dihedrals.group, dtype=np.int32):
                excluded_pairs.add(tuple(sorted((int(atom_i), int(atom_l)))))
        return excluded_pairs

    def _spacing_summary(self, snapshot, excluded_pairs: set[tuple[int, int]] | None = None) -> dict[str, float | int]:
        positions = np.asarray(snapshot.particles.position, dtype=float)
        box_lengths = np.asarray(snapshot.configuration.box[:3], dtype=float)
        tree = cKDTree(positions % box_lengths, boxsize=box_lengths)
        nearest = tree.query(positions % box_lengths, k=2, workers=-1)[0][:, 1]
        pairs_below_cutoff = tree.query_pairs(DPD_PARTICLE_SPACING_A, output_type="ndarray")
        if excluded_pairs is not None and len(pairs_below_cutoff):
            nonbonded_pairs = []
            for atom_i, atom_j in pairs_below_cutoff:
                if tuple(sorted((int(atom_i), int(atom_j)))) not in excluded_pairs:
                    nonbonded_pairs.append((int(atom_i), int(atom_j)))
            pairs_below = len(nonbonded_pairs)
        else:
            pairs_below = pairs_below_cutoff.shape[0]
        return {
            "pairs_below_cutoff": int(pairs_below),
            "min_distance_a": float(np.min(nearest)),
            "p01_distance_a": float(np.percentile(nearest, 1.0)),
            "median_distance_a": float(np.median(nearest)),
        }

    def _bond_distance_summary(self, snapshot, bond_adjacency: dict) -> dict[str, float | int]:
        positions = np.asarray(snapshot.particles.position, dtype=float)
        box_lengths = np.asarray(snapshot.configuration.box[:3], dtype=float)
        distances = []
        seen = set()
        for atom_idx, neighbors in bond_adjacency.items():
            for neighbor_idx in neighbors:
                key = tuple(sorted((atom_idx, neighbor_idx)))
                if key in seen:
                    continue
                seen.add(key)
                delta = positions[neighbor_idx] - positions[atom_idx]
                delta -= np.round(delta / box_lengths) * box_lengths
                distances.append(float(np.linalg.norm(delta)))
        distances = np.asarray(distances, dtype=float)
        return {
            "min_bond_a": float(np.min(distances)),
            "median_bond_a": float(np.median(distances)),
            "max_bond_a": float(np.max(distances)),
            "bonds_gt_max": int(np.sum(distances > DPD_MAX_BOND_DISTANCE_A)),
        }

    def _long_bond_details(self, snapshot, records: list[dict], limit: int = DPD_LONG_BOND_DETAIL_LIMIT) -> list[dict]:
        positions = np.asarray(snapshot.particles.position, dtype=float)
        box_lengths = np.asarray(snapshot.configuration.box[:3], dtype=float)
        details = []
        for record in records:
            atoms = record["atoms"]
            for local_i, local_j in record["bonds"]:
                global_i = record["local_to_global"][local_i]
                global_j = record["local_to_global"][local_j]
                delta = positions[global_j] - positions[global_i]
                delta -= np.round(delta / box_lengths) * box_lengths
                distance = float(np.linalg.norm(delta))
                if distance <= DPD_MAX_BOND_DISTANCE_A:
                    continue
                atom_i = atoms[local_i]
                atom_j = atoms[local_j]
                details.append({
                    "distance_a": distance,
                    "segment": record["segment"].label,
                    "atom_i": atom_i.label,
                    "atom_j": atom_j.label,
                    "residue_i": atom_i.metadata.get("residue_name", "UNK"),
                    "residue_j": atom_j.metadata.get("residue_name", "UNK"),
                    "repeat_i": atom_i.metadata.get("repeat_index", "?"),
                    "repeat_j": atom_j.metadata.get("repeat_index", "?"),
                    "repeat_kind_i": atom_i.metadata.get("repeat_kind", "?"),
                    "repeat_kind_j": atom_j.metadata.get("repeat_kind", "?"),
                })
        return sorted(details, key=lambda item: item["distance_a"], reverse=True)[:limit]

    def _whole_chain_movie_positions(self, snapshot, records: list[dict]) -> np.ndarray:
        """Unwrap each polymer chain for visualization without changing simulation state."""
        wrapped_positions = np.asarray(snapshot.particles.position, dtype=float)
        box_lengths = np.asarray(snapshot.configuration.box[:3], dtype=float)
        return self._whole_chain_positions_from_array(wrapped_positions, box_lengths, records)

    def _whole_chain_positions_from_array(
        self,
        wrapped_positions: np.ndarray,
        box_lengths: np.ndarray,
        records: list[dict],
    ) -> np.ndarray:
        """Unwrap each polymer chain from a wrapped coordinate array."""
        whole_positions = wrapped_positions.copy()
        for record in records:
            local_neighbors = defaultdict(set)
            for local_i, local_j in record["bonds"]:
                local_neighbors[local_i].add(local_j)
                local_neighbors[local_j].add(local_i)
            if not record["local_to_global"]:
                continue
            root_local = 0
            root_global = record["local_to_global"][root_local]
            visited = {root_local}
            stack = [root_local]
            whole_positions[root_global] = wrapped_positions[root_global]
            while stack:
                local_i = stack.pop()
                global_i = record["local_to_global"][local_i]
                for local_j in local_neighbors[local_i]:
                    if local_j in visited:
                        continue
                    global_j = record["local_to_global"][local_j]
                    delta = wrapped_positions[global_j] - wrapped_positions[global_i]
                    delta -= np.round(delta / box_lengths) * box_lengths
                    whole_positions[global_j] = whole_positions[global_i] + delta
                    visited.add(local_j)
                    stack.append(local_j)
        return whole_positions

    def _pack_whole_chains_into_pymol_cell(
        self,
        positions: np.ndarray,
        records: list[dict],
        box_lengths: np.ndarray,
    ) -> np.ndarray:
        """Convert HOOMD-centered coordinates to PyMOL's displayed 0..L cell."""
        packed_positions = positions + 0.5 * box_lengths
        for record in records:
            global_indices = record["local_to_global"]
            if not global_indices:
                continue
            centroid = packed_positions[global_indices].mean(axis=0)
            shift = np.mod(centroid, box_lengths) - centroid
            packed_positions[global_indices] += shift
        return packed_positions

    def _pymol_movie_atom_serials(self, records: list[dict]) -> dict[tuple[int, int], int]:
        """Return deterministic PDB atom serials for each record/local atom."""
        atom_serials = {}
        serial = 1
        for record_idx, record in enumerate(records):
            for local_idx, _ in enumerate(record["atoms"]):
                atom_serials[(record_idx, local_idx)] = serial
                serial += 1
            if record["atoms"]:
                serial += 1  # TER record
        return atom_serials

    def _write_pymol_conect_records(self, handle, records: list[dict]) -> None:
        """Write explicit covalent bonds so PyMOL does not infer bonds from DPD overlaps."""
        atom_serials = self._pymol_movie_atom_serials(records)
        neighbors = defaultdict(list)
        for record_idx, record in enumerate(records):
            for local_i, local_j in record["bonds"]:
                serial_i = atom_serials[(record_idx, local_i)]
                serial_j = atom_serials[(record_idx, local_j)]
                neighbors[serial_i].append(serial_j)
                neighbors[serial_j].append(serial_i)
        for serial_i in sorted(neighbors):
            bonded_serials = sorted(set(neighbors[serial_i]))
            for start in range(0, len(bonded_serials), 4):
                chunk = bonded_serials[start:start + 4]
                handle.write(f"CONECT{serial_i:5d}" + "".join(f"{serial_j:5d}" for serial_j in chunk) + "\n")

    def _write_pymol_pdb_model(self, handle, snapshot, records: list[dict], model_idx: int, step: int) -> None:
        """Append one PDB MODEL for PyMOL visualization of DPD particle motion."""
        box_lengths = np.asarray(snapshot.configuration.box[:3], dtype=float)
        positions = self._whole_chain_movie_positions(snapshot, records)
        positions = self._pack_whole_chains_into_pymol_cell(positions, records, box_lengths)
        chain_ids = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
        residue_identity = {}
        residue_counter = 0
        handle.write(f"MODEL     {model_idx:4d}\n")
        handle.write(f"REMARK DPD_STEP {step}\n")
        handle.write(f"CRYST1{box_lengths[0]:9.3f}{box_lengths[1]:9.3f}{box_lengths[2]:9.3f}  90.00  90.00  90.00 P 1           1\n")
        serial = 1
        for record_idx, record in enumerate(records):
            last_resname = "DPD"
            last_chain_id = "A"
            last_residue_number = 1
            for local_idx, atom in enumerate(record["atoms"]):
                global_idx = record["local_to_global"][local_idx]
                x, y, z = positions[global_idx]
                element = atom.element.symbol if atom.element is not None else str(atom.label)[:1]
                atom_name = str(atom.metadata.get("atom_name", f"{element}{local_idx + 1}"))[:4]
                resname = str(atom.metadata.get("residue_name", "DPD"))[:3]
                residue_key = (record_idx, atom.metadata.get("repeat_index", local_idx))
                if residue_key not in residue_identity:
                    residue_counter += 1
                    chain_idx = (residue_counter - 1) // 9999
                    if chain_idx >= len(chain_ids):
                        raise ValueError("DPD PyMOL movie exceeds single-character PDB chain ID capacity")
                    residue_identity[residue_key] = (chain_ids[chain_idx], ((residue_counter - 1) % 9999) + 1)
                chain_id, residue_number = residue_identity[residue_key]
                handle.write(
                    f"ATOM  {serial:5d} {atom_name:<4s} {resname:>3s} {chain_id:1s}"
                    f"{residue_number % 10000:4d}    {x:8.3f}{y:8.3f}{z:8.3f}"
                    f"  1.00  0.00          {element:>2s}\n"
                )
                serial += 1
                last_resname = resname
                last_chain_id = chain_id
                last_residue_number = residue_number
            if record["atoms"]:
                handle.write(f"TER   {serial:5d}      {last_resname:>3s} {last_chain_id:1s}{last_residue_number % 10000:4d}\n")
                serial += 1
        handle.write("ENDMDL\n")

    def _write_pymol_topology_cif(self, path: str, snapshot, records: list[dict]) -> None:
        """Write an mmCIF/PDBx topology frame for PyMOL/MDTraj DCD loading."""
        import openmm
        from openmm import unit as omm_unit
        from openmm.app import PDBxFile, Topology, element as omm_element

        box_lengths = np.asarray(snapshot.configuration.box[:3], dtype=float)
        positions_a = self._whole_chain_movie_positions(snapshot, records)
        positions_a = self._pack_whole_chains_into_pymol_cell(positions_a, records, box_lengths)

        topology = Topology()
        topology.setPeriodicBoxVectors((
            openmm.Vec3(float(box_lengths[0] * 0.1), 0.0, 0.0),
            openmm.Vec3(0.0, float(box_lengths[1] * 0.1), 0.0),
            openmm.Vec3(0.0, 0.0, float(box_lengths[2] * 0.1)),
        ) * omm_unit.nanometer)
        positions_nm = []
        atom_lookup = {}

        for record_idx, record in enumerate(records):
            chain = topology.addChain(id=str(record_idx + 1))
            residues = {}
            for local_idx, atom in enumerate(record["atoms"]):
                residue_key = atom.metadata.get("repeat_index", local_idx)
                try:
                    residue_id = str(int(residue_key) + 1)
                except (TypeError, ValueError):
                    residue_id = str(local_idx + 1)
                residue = residues.get(residue_key)
                if residue is None:
                    resname = str(atom.metadata.get("residue_name", "DPD"))
                    residue = topology.addResidue(resname, chain, id=residue_id)
                    residues[residue_key] = residue
                symbol = atom.element.symbol if atom.element is not None else str(atom.label)[:1]
                atom_name = str(atom.metadata.get("atom_name", f"{symbol}{local_idx + 1}"))
                atom_lookup[(record_idx, local_idx)] = topology.addAtom(
                    atom_name,
                    omm_element.get_by_symbol(symbol),
                    residue,
                )
                global_idx = record["local_to_global"][local_idx]
                x, y, z = positions_a[global_idx] * 0.1
                positions_nm.append(openmm.Vec3(float(x), float(y), float(z)))
            for local_i, local_j in record["bonds"]:
                topology.addBond(atom_lookup[(record_idx, local_i)], atom_lookup[(record_idx, local_j)])

        with open(path, "w") as handle:
            PDBxFile.writeFile(topology, positions_nm * omm_unit.nanometer, handle)

    def _postprocess_pymol_dcd(self, raw_dcd_path: str, topology_path: str, processed_dcd_path: str, records: list[dict]) -> int:
        """Apply whole-chain/PyMOL-cell transforms to a raw HOOMD DCD using MDTraj."""
        import mdtraj as md

        trajectory = md.load_dcd(raw_dcd_path, top=topology_path)
        if DPD_PYMOL_MAX_FRAMES is not None and trajectory.n_frames > DPD_PYMOL_MAX_FRAMES:
            trajectory = trajectory[:DPD_PYMOL_MAX_FRAMES]
        for frame_idx in range(trajectory.n_frames):
            box_lengths = trajectory.unitcell_lengths[frame_idx]
            positions = self._whole_chain_positions_from_array(trajectory.xyz[frame_idx], box_lengths, records)
            trajectory.xyz[frame_idx] = self._pack_whole_chains_into_pymol_cell(positions, records, box_lengths)
        trajectory.save_dcd(processed_dcd_path)
        return int(trajectory.n_frames)

    def _maybe_write_pymol_frame(
        self,
        handle,
        snapshot,
        records: list[dict],
        frame_state: dict[str, int],
        step: int,
        force: bool = False,
    ) -> None:
        if handle is None:
            return
        if frame_state["frames"] >= DPD_PYMOL_MAX_FRAMES:
            return
        if not force and step - frame_state["last_step"] < DPD_PYMOL_FRAME_INTERVAL_STEPS:
            return
        frame_state["frames"] += 1
        frame_state["last_step"] = step
        self._write_pymol_pdb_model(handle, snapshot, records, frame_state["frames"], step)

    def _close_contact_details(
        self,
        snapshot,
        records: list[dict],
        excluded_pairs: set[tuple[int, int]],
        limit: int = DPD_CLOSE_CONTACT_DETAIL_LIMIT,
    ) -> list[dict]:
        positions = np.asarray(snapshot.particles.position, dtype=float)
        box_lengths = np.asarray(snapshot.configuration.box[:3], dtype=float)
        tree = cKDTree(positions % box_lengths, boxsize=box_lengths)
        pairs = tree.query_pairs(DPD_PARTICLE_SPACING_A, output_type="ndarray")
        global_atoms = {}
        for record in records:
            for local_idx, atom in enumerate(record["atoms"]):
                global_atoms[record["local_to_global"][local_idx]] = (record["segment"], atom)
        details = []
        for atom_i, atom_j in pairs:
            key = tuple(sorted((int(atom_i), int(atom_j))))
            if key in excluded_pairs:
                continue
            delta = positions[int(atom_j)] - positions[int(atom_i)]
            delta -= np.round(delta / box_lengths) * box_lengths
            segment_i, atom_i_obj = global_atoms[int(atom_i)]
            segment_j, atom_j_obj = global_atoms[int(atom_j)]
            details.append({
                "distance_a": float(np.linalg.norm(delta)),
                "segment_i": segment_i.label,
                "segment_j": segment_j.label,
                "atom_i": atom_i_obj.label,
                "atom_j": atom_j_obj.label,
                "repeat_i": atom_i_obj.metadata.get("repeat_index", "?"),
                "repeat_j": atom_j_obj.metadata.get("repeat_index", "?"),
                "repeat_kind_i": atom_i_obj.metadata.get("repeat_kind", "?"),
                "repeat_kind_j": atom_j_obj.metadata.get("repeat_kind", "?"),
            })
        return sorted(details, key=lambda item: item["distance_a"])[:limit]

    def _geometry_converged(self, snapshot, bond_adjacency: dict, excluded_pairs: set[tuple[int, int]]) -> bool:
        spacing = self._spacing_summary(snapshot, excluded_pairs)
        bonds = self._bond_distance_summary(snapshot, bond_adjacency)
        return spacing["pairs_below_cutoff"] == 0 and bonds["bonds_gt_max"] == 0

    def _unwrap_positions(self, wrapped_positions: np.ndarray, chain_atom_indices: list[list[int]], bond_adjacency: dict, box_length: float) -> np.ndarray:
        box_lengths = np.array([box_length, box_length, box_length], dtype=float)
        unwrapped = wrapped_positions.copy()
        for chain_indices in chain_atom_indices:
            chain_set = set(chain_indices)
            root = chain_indices[0]
            visited = {root}
            stack = [root]
            while stack:
                atom_idx = stack.pop()
                for neighbor_idx in bond_adjacency[atom_idx]:
                    if neighbor_idx in visited or neighbor_idx not in chain_set:
                        continue
                    delta = wrapped_positions[neighbor_idx] - wrapped_positions[atom_idx]
                    delta -= np.round(delta / box_lengths) * box_lengths
                    unwrapped[neighbor_idx] = unwrapped[atom_idx] + delta
                    visited.add(neighbor_idx)
                    stack.append(neighbor_idx)
        return unwrapped

    def place(
        self,
        polymer_root: Primitive,
        polymer_chain_residues: list[list[Primitive]],
        resname_map: dict[str, str],
        output_name: str | None = None,
        total_mass_da: float | None = None,
        target_density_g_cm3: float | None = None,
    ) -> list[float]:
        records = self._segment_records(polymer_root)
        atom_count = sum(len(record["atoms"]) for record in records)
        box_length = self._box_length(atom_count, total_mass_da=total_mass_da, target_density_g_cm3=target_density_g_cm3)
        if DIAGNOSTIC_MODE:
            density_text = "unknown" if total_mass_da is None else f"{cubic_density_g_cm3(total_mass_da, box_length):.3f} g/cm^3"
            print(
                f"DPD diagnostic: records={len(records)}, atoms={atom_count}, box={box_length:.2f} A "
                f"({box_length * 0.1:.2f} nm), density={density_text}, initializer={DPD_INITIALIZATION_MODE}"
            )
        init_start_s = time.perf_counter()
        if DPD_INITIALIZATION_MODE == "repeat_unit_random_walk":
            self._initialize_repeat_unit_random_walk(polymer_chain_residues, box_length)
        elif DPD_INITIALIZATION_MODE == "chains":
            self._initialize_polymer_positions(polymer_chain_residues, box_length)
        else:
            raise ValueError(f"Unknown DPD_INITIALIZATION_MODE={DPD_INITIALIZATION_MODE!r}")
        if DIAGNOSTIC_MODE:
            print(f"DPD diagnostic: initialization took {time.perf_counter() - init_start_s:.2f} s")
        labels, off_unit = self._openff_labels(polymer_root, resname_map)
        tables = self._parameter_tables(records, labels, off_unit)
        frame, chain_atom_indices, bond_adjacency = self._build_frame(records, tables, box_length)
        excluded_pairs = self._excluded_pairs_from_frame(frame)

        integrator = hoomd.md.Integrator(dt=DPD_DT)
        integrator.methods.append(hoomd.md.methods.ConstantVolume(filter=hoomd.filter.All()))
        bond_force = hoomd.md.bond.Harmonic()
        for bond_type, params in tables["bond_params"].items():
            bond_force.params[bond_type] = params
        angle_force = hoomd.md.angle.Harmonic()
        for angle_type, params in tables["angle_params"].items():
            angle_force.params[angle_type] = params
        dihedral_force = hoomd.md.dihedral.Periodic()
        for dihedral_type, params in tables["dihedral_params"].items():
            dihedral_force.params[dihedral_type] = params
        nlist = hoomd.md.nlist.Cell(buffer=0.4, exclusions=DPD_NLIST_EXCLUSIONS)
        dpd_force = hoomd.md.pair.DPD(nlist=nlist, default_r_cut=DPD_R_CUT_A, kT=DPD_KT_KCAL_MOL)
        for pair, params in tables["pair_params"].items():
            dpd_force.params[pair] = params
        integrator.forces.extend([bond_force, angle_force, dihedral_force, dpd_force])

        hoomd_device = make_hoomd_device()
        simulation = hoomd.Simulation(device=hoomd_device, seed=int(self.rng.integers(1, 65000)))
        simulation.operations.integrator = integrator
        simulation.create_state_from_snapshot(frame)
        if output_name is not None and WRITE_DPD_GSD:
            with gsd.hoomd.open(name=f"{output_name}_aa_init.gsd", mode="w") as handle:
                handle.append(frame)
            simulation.operations.writers.append(hoomd.write.GSD(trigger=hoomd.trigger.Periodic(DPD_REPORT_INTERVAL), filename=f"{output_name}_aa_traj.gsd"))
        pymol_topology_path = None
        raw_pymol_movie_path = None
        pymol_movie_path = None
        pymol_script_path = None
        raw_dcd_writer = None
        if output_name is not None and WRITE_DPD_PYMOL_MOVIE:
            pymol_topology_path = f"{output_name}_aa_dpd_movie_topology.cif"
            raw_pymol_movie_path = f"{output_name}_aa_dpd_movie_raw.dcd"
            pymol_movie_path = f"{output_name}_aa_dpd_movie.dcd"
            pymol_script_path = f"{output_name}_aa_dpd_movie.pml"
            self._write_pymol_topology_cif(pymol_topology_path, frame, records)
            raw_dcd_writer = hoomd.write.DCD(
                    trigger=hoomd.trigger.Periodic(DPD_PYMOL_FRAME_INTERVAL_STEPS),
                    filename=raw_pymol_movie_path,
                    overwrite=True,
                    unwrap_full=False,
            )
            simulation.operations.writers.append(raw_dcd_writer)

        if DIAGNOSTIC_MODE:
            print(f"DPD diagnostic: HOOMD device = {type(hoomd_device).__name__}")
            print(f"DPD diagnostic: initial spacing = {self._spacing_summary(frame, excluded_pairs)}")
            initial_bond_summary = self._bond_distance_summary(frame, bond_adjacency)
            print(f"DPD diagnostic: initial bonds = {initial_bond_summary}")
            if initial_bond_summary["bonds_gt_max"] > 0:
                print(f"DPD diagnostic: initial long bond details = {self._long_bond_details(frame, records)}")
        simulation.run(1)
        total_steps = 1
        start = time.perf_counter()
        bond_plateau_intervals = 0
        spacing_plateau_intervals = 0
        next_report_step = total_steps + DPD_STEPS_PER_INTERVAL
        report_start_s = start
        best_bonds_gt_max = self._bond_distance_summary(simulation.state.get_snapshot(), bond_adjacency)["bonds_gt_max"]
        best_pairs_below_cutoff = self._spacing_summary(simulation.state.get_snapshot(), excluded_pairs)["pairs_below_cutoff"]
        while not self._geometry_converged(simulation.state.get_snapshot(), bond_adjacency, excluded_pairs):
            if DPD_STEPS_MAX is not None and total_steps >= DPD_STEPS_MAX:
                break
            steps = max(1, next_report_step - total_steps)
            if DPD_STEPS_MAX is not None:
                steps = min(steps, DPD_STEPS_MAX - total_steps)
            interval_start_s = time.perf_counter()
            simulation.run(steps)
            total_steps += steps
            interval_snapshot = simulation.state.get_snapshot()
            spacing_summary = self._spacing_summary(interval_snapshot, excluded_pairs)
            bond_summary = self._bond_distance_summary(interval_snapshot, bond_adjacency)
            if bond_summary["bonds_gt_max"] <= best_bonds_gt_max - DPD_PLATEAU_MIN_BOND_IMPROVEMENT:
                best_bonds_gt_max = bond_summary["bonds_gt_max"]
                bond_plateau_intervals = 0
            else:
                bond_plateau_intervals += 1
            if spacing_summary["pairs_below_cutoff"] < best_pairs_below_cutoff:
                best_pairs_below_cutoff = spacing_summary["pairs_below_cutoff"]
                spacing_plateau_intervals = 0
            else:
                spacing_plateau_intervals += 1
            report_due = total_steps >= next_report_step or self._geometry_converged(interval_snapshot, bond_adjacency, excluded_pairs)
            if DIAGNOSTIC_MODE and report_due:
                print(
                    f"DPD diagnostic: steps={total_steps}/{DPD_STEPS_MAX if DPD_STEPS_MAX is not None else 'until converged'}, "
                    f"interval_s={time.perf_counter() - report_start_s:.2f}, "
                    f"spacing={spacing_summary}, "
                    f"bonds={bond_summary}"
                )
                if (
                    bond_summary["bonds_gt_max"] > 0
                    and DPD_PLATEAU_WINDOW_INTERVALS > 0
                    and bond_plateau_intervals > 0
                    and bond_plateau_intervals % DPD_PLATEAU_WINDOW_INTERVALS == 0
                ):
                    print(
                        f"DPD diagnostic: long-bond plateau warning after {bond_plateau_intervals} intervals without improvement; "
                        f"best_bonds_gt_max={best_bonds_gt_max}, current_long_bond_details={self._long_bond_details(interval_snapshot, records)}"
                    )
            if report_due:
                next_report_step = total_steps + DPD_STEPS_PER_INTERVAL
                report_start_s = time.perf_counter()
                if (
                    spacing_summary["pairs_below_cutoff"] > 0
                    and DPD_PLATEAU_WINDOW_INTERVALS > 0
                    and spacing_plateau_intervals > 0
                    and spacing_plateau_intervals % DPD_PLATEAU_WINDOW_INTERVALS == 0
                ):
                    print(
                        f"DPD diagnostic: close-contact plateau warning after {spacing_plateau_intervals} intervals without improvement; "
                        f"best_pairs_below_cutoff={best_pairs_below_cutoff}, current_close_contact_details={self._close_contact_details(interval_snapshot, records, excluded_pairs)}"
                    )
        elapsed = time.perf_counter() - start
        snapshot = simulation.state.get_snapshot()
        pymol_movie_frames = 0
        if pymol_movie_path is not None:
            simulation.operations.writers.remove(raw_dcd_writer)
            pymol_movie_frames = self._postprocess_pymol_dcd(
                raw_pymol_movie_path,
                pymol_topology_path,
                pymol_movie_path,
                records,
            )
            with open(pymol_script_path, "w") as script_handle:
                script_handle.write("set connect_mode, 1\n")
                script_handle.write(f"load {Path(pymol_topology_path).name}, dpd_movie\n")
                script_handle.write(f"load_traj {Path(pymol_movie_path).name}, dpd_movie\n")
                script_handle.write("hide everything, dpd_movie\n")
                script_handle.write("show sticks, dpd_movie\n")
                script_handle.write("color green, elem C and dpd_movie\n")
                script_handle.write("color red, elem O and dpd_movie\n")
                script_handle.write("color gray70, elem H and dpd_movie\n")
                script_handle.write("set stick_radius, 0.08, dpd_movie\n")
                script_handle.write("set connect_mode, 0\n")
            if DIAGNOSTIC_MODE:
                print(f"DPD diagnostic: wrote PyMOL topology mmCIF to {pymol_topology_path}")
                print(f"DPD diagnostic: wrote processed PyMOL DCD with {pymol_movie_frames} frames to {pymol_movie_path}")
                print(f"DPD diagnostic: wrote PyMOL loader script to {pymol_script_path}")
        geometry_converged = self._geometry_converged(snapshot, bond_adjacency, excluded_pairs)
        if DIAGNOSTIC_MODE:
            print(f"DPD diagnostic: geometry_converged={geometry_converged}, elapsed_s={elapsed:.2f}")
            if not geometry_converged:
                print(f"DPD diagnostic: final long bond details = {self._long_bond_details(snapshot, records)}")
                print(f"DPD diagnostic: final close contact details = {self._close_contact_details(snapshot, records, excluded_pairs)}")
        if DPD_REQUIRE_GEOMETRY_CONVERGENCE and DPD_STEPS_MAX is not None and not geometry_converged:
            raise RuntimeError(
                f"DPD geometry did not converge after {total_steps} steps: "
                f"spacing={self._spacing_summary(snapshot, excluded_pairs)}, bonds={self._bond_distance_summary(snapshot, bond_adjacency)}"
            )
        final_positions = self._unwrap_positions(np.asarray(snapshot.particles.position, dtype=float), chain_atom_indices, bond_adjacency, box_length)

        for record in records:
            for local_idx, atom in enumerate(record["atoms"]):
                atom.shape = PointCloud(final_positions[record["local_to_global"][local_idx]])
        for residues in polymer_chain_residues:
            for residue in residues:
                residue.shape = PointCloud(np.vstack([np.asarray(atom.shape.centroid, dtype=float) for atom in residue.children]))

        self.summary = {
            "atoms": atom_count,
            "device": DPD_DEVICE,
            "initialization_mode": DPD_INITIALIZATION_MODE,
            "bonds": int(frame.bonds.N),
            "angles": int(frame.angles.N),
            "dihedrals": int(frame.dihedrals.N),
            "particle_types": len(tables["particle_types"]),
            "steps": total_steps,
            "elapsed_s": elapsed,
            "box_length_a": box_length,
            "target_density_g_cm3": target_density_g_cm3,
            "total_mass_da": total_mass_da,
            "initial_density_g_cm3": None if total_mass_da is None else cubic_density_g_cm3(total_mass_da, box_length),
            "geometry_converged": geometry_converged,
            "pymol_topology_path": pymol_topology_path,
            "raw_pymol_movie_path": raw_pymol_movie_path,
            "pymol_movie_path": pymol_movie_path,
            "pymol_script_path": pymol_script_path,
            "pymol_movie_frames": pymol_movie_frames,
        }
        return [box_length, box_length, box_length, 0.0, 0.0, 0.0]


def build_ionomer_system(system_name: str, n_chains: int, random_seed: int) -> BuiltIonomerSystem:
    """Build one all-atom PE/PEAA ionomer system with DPD-placed polymers."""
    rng = np.random.default_rng(random_seed)
    np.random.seed(random_seed)
    lexicon = build_repeat_lexicon()
    sequence_template = expanded_sequence(system_name)

    universe = Primitive(label=f"ionomer_{system_name}", role=PrimitiveRole.UNIVERSE)
    placed_with_dpd = Primitive(label="placed_with_DPD")
    not_placed_with_dpd = Primitive(label="not_placed_with_DPD")
    universe.metadata.update({
        "system_name": system_name,
        "n_chains": str(n_chains),
        "terminal_caps": "explicit_hydrogen_residues",
        "placement_method": "AllAtomDPDPlacement",
        "dpd_openff_force_field": DPD_OPENFF_FORCE_FIELD,
        "dpd_atom_number_density_a3": str(DPD_ATOM_NUMBER_DENSITY_A3),
        "dpd_particle_spacing_a": str(DPD_PARTICLE_SPACING_A),
    })
    placed_with_dpd.metadata["placement_method"] = "AllAtomDPDPlacement"
    not_placed_with_dpd.metadata["placement_method"] = "post_DPD_geometry"
    universe.attach_child(placed_with_dpd)

    chain_sequences = []
    polymer_chain_residues = []
    sodium_count = 0

    for chain_idx in tqdm(range(n_chains), desc=f"building {system_name} chains", unit="chain"):
        segment = Primitive(label=f"chain_{chain_idx:04d}", role=PrimitiveRole.SEGMENT)
        segment.metadata.update({"system_name": system_name, "chain_index": str(chain_idx)})
        residue_handles = []
        real_residues = []
        chain_sequences.append(sequence_template)

        for repeat_idx, repeat_kind in enumerate(sequence_template):
            residue = lexicon[repeat_kind].copy()
            residue.label = f"repeat_{repeat_idx:03d}_{repeat_kind}"
            residue.role = PrimitiveRole.RESIDUE
            residue.metadata.update({
                "system_name": system_name,
                "chain_index": str(chain_idx),
                "repeat_index": str(repeat_idx),
                "repeat_kind": repeat_kind,
                "residue_name": RESNAME_MAP[repeat_kind],
            })
            for atom in residue.children:
                atom.role = PrimitiveRole.PARTICLE
                atom.metadata.update({
                    "system_name": system_name,
                    "chain_index": str(chain_idx),
                    "repeat_index": str(repeat_idx),
                    "repeat_kind": repeat_kind,
                    "residue_name": RESNAME_MAP[repeat_kind],
                })
            residue_handles.append(segment.attach_child(residue))
            real_residues.append(residue)

        segment.set_topology(
            nx.path_graph(residue_handles, create_using=TopologicalStructure),
            max_registration_iter=100,
        )
        placed_with_dpd.attach_child(segment)
        polymer_chain_residues.append(real_residues)

    dpd_output_name = None
    if WRITE_DPD_GSD or WRITE_DPD_PYMOL_MOVIE:
        dpd_dir = OUTPUT_ROOT / system_name / "dpd"
        dpd_dir.mkdir(parents=True, exist_ok=True)
        dpd_output_name = str(dpd_dir / f"{system_name}_polymer_dpd")
    resname_map = {residue.label: residue.metadata.get("residue_name", "UNK") for residues in polymer_chain_residues for residue in residues}
    polymer_mass_da = primitive_mass_da(placed_with_dpd)
    sodium_mass_da = n_chains * sequence_template.count("PEAA") * primitive_mass_da(lexicon["NA"])
    total_mass_da = polymer_mass_da + sodium_mass_da
    if DIAGNOSTIC_MODE:
        density_box_a = density_box_length_a(total_mass_da, TARGET_INITIAL_DENSITY_G_CM3)
        print(
            f"{system_name}: density-first mass polymer={polymer_mass_da:.1f} Da, "
            f"Na={sodium_mass_da:.1f} Da, total={total_mass_da:.1f} Da, "
            f"target box={density_box_a:.2f} A ({density_box_a * 0.1:.2f} nm)"
        )
    universe.metadata.update({
        "target_initial_density_g_cm3": str(TARGET_INITIAL_DENSITY_G_CM3),
        "density_first_total_mass_da": str(total_mass_da),
        "density_first_polymer_mass_da": str(polymer_mass_da),
        "density_first_sodium_mass_da": str(sodium_mass_da),
    })
    dpd_placement = AllAtomDPDPlacement(random_seed=random_seed)
    box_parameters = dpd_placement.place(
        universe,
        polymer_chain_residues,
        resname_map=resname_map,
        output_name=dpd_output_name,
        total_mass_da=total_mass_da,
        target_density_g_cm3=TARGET_INITIAL_DENSITY_G_CM3,
    )
    universe.metadata["unit_cell_parameters"] = box_parameters
    universe.metadata["all_atom_dpd_summary"] = json.dumps(dpd_placement.summary)
    print(f"{system_name}: all-atom DPD summary = {dpd_placement.summary}")

    box_lengths = np.asarray(box_parameters[:3], dtype=float)
    polymer_atoms = [atom for residues in polymer_chain_residues for residue in residues for atom in residue.children]
    polymer_tree = cKDTree(wrapped_positions_from_primitives(polymer_atoms, box_lengths), boxsize=box_lengths)
    placed_sodium_positions = []

    for chain_idx, residues in enumerate(polymer_chain_residues):
        for repeat_idx, (repeat_kind, peaa_residue) in enumerate(zip(sequence_template, residues)):
            if repeat_kind != "PEAA":
                continue
            sodium = lexicon["NA"].copy()
            sodium.label = f"sodium_chain{chain_idx:04d}_repeat{repeat_idx:03d}"
            sodium.role = PrimitiveRole.RESIDUE
            sodium_position = place_sodium_near_peaa(
                peaa_residue,
                rng,
                polymer_tree=polymer_tree,
                placed_sodium_positions=placed_sodium_positions,
                box_lengths=box_lengths,
            )
            set_single_atom_residue_position(sodium, sodium_position)
            placed_sodium_positions.append(np.asarray(sodium_position, dtype=float) % box_lengths)
            sodium.metadata.update({
                "system_name": system_name,
                "chain_index": str(chain_idx),
                "repeat_index": str(repeat_idx),
                "associated_peaa_residue": f"repeat_{repeat_idx:03d}_PEAA",
                "residue_name": RESNAME_MAP["NA"],
            })
            for atom in sodium.children:
                atom.role = PrimitiveRole.PARTICLE
                atom.metadata.update(sodium.metadata)

            sodium_segment = Primitive(
                label=f"sodium_chain{chain_idx:04d}_repeat{repeat_idx:03d}",
                role=PrimitiveRole.SEGMENT,
            )
            sodium_segment.metadata.update(sodium.metadata)
            sodium_segment.attach_child(sodium)
            not_placed_with_dpd.attach_child(sodium_segment)
            sodium_count += 1

    if not_placed_with_dpd.children:
        universe.attach_child(not_placed_with_dpd)

    return BuiltIonomerSystem(
        primitive=universe,
        system_name=system_name,
        chain_sequences=chain_sequences,
        sodium_count=sodium_count,
        sdf_paths=[],
    )


## 4. Build Systems

For production-size builds, run one system at a time unless you know your workstation has enough memory.

In [27]:
build_start_s = time.perf_counter()
built_systems = []
for system_name in tqdm(SELECTED_SYSTEMS, desc="building selected systems", unit="system"):
    built = build_ionomer_system(system_name, n_chains=N_CHAINS, random_seed=RANDOM_SEED)
    built_systems.append(built)

    sequence = expanded_sequence(system_name)
    print(f"{system_name}: chains={N_CHAINS}")
    print(f"  repeat units per chain: {len(sequence)}")
    print(f"  PEAA per chain: {sequence.count('PEAA')}")
    print(f"  sodium ions: {built.sodium_count}")
    print(f"  total particles: {len(built.primitive.leaves)}")

build_elapsed_s = time.perf_counter() - build_start_s
print(f"Coordinate-generation wall time: {build_elapsed_s:.2f} s ({build_elapsed_s / 60.0:.2f} min)")

building selected systems:   0%|          | 0/1 [00:00<?, ?system/s]

building m5 chains:   0%|          | 0/200 [00:00<?, ?chain/s]

Attempting to infer internal connections automatically from given topology; user should verify the connections assigned make sense!
Attempting to infer internal connections automatically from given topology; user should verify the connections assigned make sense!
Attempting to infer internal connections automatically from given topology; user should verify the connections assigned make sense!
Attempting to infer internal connections automatically from given topology; user should verify the connections assigned make sense!
Attempting to infer internal connections automatically from given topology; user should verify the connections assigned make sense!
Attempting to infer internal connections automatically from given topology; user should verify the connections assigned make sense!
Attempting to infer internal connections automatically from given topology; user should verify the connections assigned make sense!
Attempting to infer internal connections automatically from given topology; 

m5: all-atom DPD summary = {'atoms': 70400, 'bonds': 70200, 'angles': 138600, 'dihedrals': 265800, 'particle_types': 4, 'steps': 1001, 'elapsed_s': 90.52218160600023, 'box_length_a': 141.21393341304253}
m5: chains=200
  repeat units per chain: 35
  PEAA per chain: 7
  sodium ions: 1400
  total particles: 71800
Coordinate-generation wall time: 120.23 s (2.00 min)


## 5. Coordinate Diagnostics

The starting coordinates are not a melt-packing algorithm. They are non-overlapping DPD-initialized coordinates intended for OpenMM minimization and vacuum collapse before optional periodic NPT.

In [28]:
def primitive_positions(primitive: Primitive) -> np.ndarray:
    """Collect leaf-particle coordinates from a MuPT primitive."""
    return np.vstack([np.array(leaf.shape.centroid, dtype=float) for leaf in primitive.leaves if leaf.shape is not None])


def minimum_pair_distance(positions: np.ndarray) -> float:
    """Return nearest-neighbor distance without allocating an O(N^2) matrix."""
    from scipy.spatial import cKDTree

    distances, _ = cKDTree(positions).query(positions, k=2)
    return float(np.min(distances[:, 1]))


def role_counts(primitive: Primitive) -> dict[str, int]:
    """Return counts of assigned roles in a Primitive hierarchy."""
    return dict(Counter(node.role.value for node in PreOrderIter(primitive)))


def validate_particle_coordinates(primitive: Primitive) -> None:
    """Assert all exported particles have finite assigned coordinates."""
    missing = []
    nonfinite = []
    for leaf in primitive.leaves:
        if leaf.role != PrimitiveRole.PARTICLE:
            continue
        if leaf.shape is None:
            missing.append(leaf.label)
            continue
        position = np.array(leaf.shape.centroid, dtype=float)
        if not np.all(np.isfinite(position)):
            nonfinite.append(leaf.label)
    if missing:
        raise ValueError(f"Particle leaves missing coordinates: {missing[:10]}")
    if nonfinite:
        raise ValueError(f"Particle leaves with non-finite coordinates: {nonfinite[:10]}")


def primitive_bond_distances(primitive: Primitive) -> np.ndarray:
    """Return all covalent bond distances from MuPT connector topology."""
    index = build_saamr_role_topology_index(primitive)
    endpoint_cache = {}
    distances = []
    seen = set()
    for node in index.bond_nodes:
        for conn_pair in node.internal_connections:
            conn_refs = tuple(sorted(conn_pair, key=lambda ref: (repr(ref.primitive_handle), repr(ref.connector_handle))))
            atom_1 = resolve_to_atom_cached(node, conn_refs[0], endpoint_cache)
            atom_2 = resolve_to_atom_cached(node, conn_refs[1], endpoint_cache)
            bond_key = tuple(sorted((id(atom_1), id(atom_2))))
            if bond_key in seen:
                continue
            seen.add(bond_key)
            distances.append(float(np.linalg.norm(np.asarray(atom_2.shape.centroid, dtype=float) - np.asarray(atom_1.shape.centroid, dtype=float))))
    return np.asarray(distances, dtype=float)


def load_rdkit_sdf_records(paths: list[Path]) -> list[Chem.Mol]:
    """Load every SDF molecule record without stripping explicit hydrogens."""
    records = []
    for path in paths:
        supplier = Chem.SDMolSupplier(str(path), removeHs=False, sanitize=False)
        for record_idx, mol in enumerate(supplier):
            if mol is None:
                raise ValueError(f"Could not read record {record_idx} from {path}")
            sanitized = Chem.Mol(mol)
            Chem.SanitizeMol(sanitized)
            records.append(sanitized)
    return records


for built in built_systems:
    validate_particle_coordinates(built.primitive)
    positions = primitive_positions(built.primitive)
    span = positions.max(axis=0) - positions.min(axis=0)
    min_distance = minimum_pair_distance(positions)
    bond_distances = primitive_bond_distances(built.primitive)
    print(f"{built.system_name}: role counts = {role_counts(built.primitive)}")
    print(f"{built.system_name}: minimum atom distance = {min_distance:.3f} A")
    print(f"{built.system_name}: bonded distances min/median/max = {bond_distances.min():.3f}/{np.median(bond_distances):.3f}/{bond_distances.max():.3f} A")
    print(f"{built.system_name}: bonds > 3 A = {int(np.sum(bond_distances > 3.0))}")
    print(f"{built.system_name}: coordinate span = {span[0]:.1f} x {span[1]:.1f} x {span[2]:.1f} A")
    if min_distance < MIN_ALLOWED_DISTANCE_A:
        print(
            f"Warning: {built.system_name} is below the requested initial-distance threshold. "
            "Continuing to OpenMM minimization/collapse; inspect the structure if minimization fails."
        )


m5: role counts = {'universe': 1, 'unassigned': 2, 'segment': 1600, 'residue': 8400, 'particle': 71800}
m5: minimum atom distance = 0.545 A
m5: bonded distances min/median/max = 0.806/1.493/3.610 A
m5: bonds > 3 A = 449
m5: coordinate span = 462.7 x 445.4 x 438.0 A


## 6. Export Role-Aware SDF Files

Each polymer chain and sodium ion is still exported as its own SDF molecule record, but the notebook now stores those records in one multi-record SDF file per system instead of one file per segment. Sodium residues remain associated with their PEAA repeat through metadata, not covalent bonds.

A single multi-conformer SDF entry is not enough for the repeated sodium ions in this workflow: OpenFF treats multiple conformers as alternate coordinates for one chemical graph, not as distinct molecule instances with different residue numbers. That means multi-record SDF is the practical file-count reduction here, while true conformer-sharing would still need a separate per-instance coordinate/metadata transport layer.

In [29]:
MUPT_ATOM_PROPS_FOR_SDF = [
    # RDKit SDF reload does not preserve AtomPDBResidueInfo directly, so write
    # both PDB-style atom metadata and MuPT hierarchy metadata as atom-property
    # lists for downstream OpenFF/OpenMM notebooks.
    "chain_id",
    "residue_id",
    "residue_name",
    "mupt_segment_index",
    "mupt_segment_label",
    "mupt_residue_index",
    "mupt_residue_label",
    "mupt_particle_index",
    "mupt_particle_label",
]


def prepare_mupt_sdf_atom_props(mol: Chem.Mol) -> None:
    """Store MuPT atom props as SDF atom-property lists before writing."""
    for prop_name in MUPT_ATOM_PROPS_FOR_SDF:
        Chem.CreateAtomStringPropertyList(mol, prop_name)


def segment_label_from_mol(mol: Chem.Mol, mol_idx: int) -> str:
    """Return the MuPT segment label stored on an exported RDKit molecule."""
    if mol.HasProp("mupt_segment_label"):
        return mol.GetProp("mupt_segment_label")
    if mol.GetNumAtoms() and mol.GetAtomWithIdx(0).HasProp("mupt_segment_label"):
        return mol.GetAtomWithIdx(0).GetProp("mupt_segment_label")
    return f"mol_{mol_idx:05d}"


def make_rdkit_molecule_whole(mol: Chem.Mol, box_parameters: list[float] | tuple[float, ...] | None) -> None:
    """Unwrap bonded atoms across an orthorhombic periodic box in-place."""
    if box_parameters is None or mol.GetNumAtoms() <= 1:
        return
    box_lengths = np.asarray(box_parameters[:3], dtype=float)
    if box_lengths.shape != (3,) or np.any(box_lengths <= 0) or not np.all(np.isfinite(box_lengths)):
        return

    conf = mol.GetConformer()
    wrapped_positions = np.asarray(conf.GetPositions(), dtype=float)
    unwrapped_positions = wrapped_positions.copy()
    adjacency = [[] for _ in range(mol.GetNumAtoms())]
    for bond in mol.GetBonds():
        begin_idx = bond.GetBeginAtomIdx()
        end_idx = bond.GetEndAtomIdx()
        adjacency[begin_idx].append(end_idx)
        adjacency[end_idx].append(begin_idx)

    visited = np.zeros(mol.GetNumAtoms(), dtype=bool)
    for root_idx in range(mol.GetNumAtoms()):
        if visited[root_idx]:
            continue
        visited[root_idx] = True
        stack = [root_idx]
        while stack:
            atom_idx = stack.pop()
            for neighbor_idx in adjacency[atom_idx]:
                if visited[neighbor_idx]:
                    continue
                delta = wrapped_positions[neighbor_idx] - wrapped_positions[atom_idx]
                delta -= np.round(delta / box_lengths) * box_lengths
                unwrapped_positions[neighbor_idx] = unwrapped_positions[atom_idx] + delta
                visited[neighbor_idx] = True
                stack.append(neighbor_idx)

    for atom_idx, position in enumerate(unwrapped_positions):
        conf.SetAtomPosition(atom_idx, Point3D(float(position[0]), float(position[1]), float(position[2])))


def write_rdkit_mols_to_pdbx(rdkit_mols: list[Chem.Mol], output_path: Path) -> None:
    """Write all RDKit molecule records as one visualization-friendly mmCIF/PDBx file."""
    import openmm
    from openmm import unit as omm_unit
    from openmm.app import PDBxFile, Topology, element

    topology = Topology()
    positions = []

    for mol_idx, mol in enumerate(rdkit_mols):
        chain = topology.addChain(segment_label_from_mol(mol, mol_idx))
        residues = {}
        atom_lookup = {}
        conf = mol.GetConformer()

        for atom_idx, atom in enumerate(mol.GetAtoms()):
            residue_name = str(atom.GetProp("residue_name")) if atom.HasProp("residue_name") else "UNK"
            residue_id = str(atom.GetProp("residue_id")) if atom.HasProp("residue_id") else str(mol_idx + 1)
            residue_key = (residue_id, residue_name)
            residue = residues.get(residue_key)
            if residue is None:
                residue = topology.addResidue(residue_name, chain, id=residue_id)
                residues[residue_key] = residue

            symbol = atom.GetSymbol()
            atom_lookup[atom_idx] = topology.addAtom(f"{symbol}{atom_idx + 1}", element.get_by_symbol(symbol), residue)
            position = conf.GetAtomPosition(atom_idx)
            positions.append(openmm.Vec3(position.x * 0.1, position.y * 0.1, position.z * 0.1))

        for bond in mol.GetBonds():
            topology.addBond(atom_lookup[bond.GetBeginAtomIdx()], atom_lookup[bond.GetEndAtomIdx()])

    with output_path.open("w") as handle:
        PDBxFile.writeFile(topology, positions * omm_unit.nanometer, handle)


for idx, built in enumerate(tqdm(built_systems, desc="exporting systems", unit="system")):
    sdf_dir = OUTPUT_ROOT / built.system_name / "sdf"
    sdf_dir.mkdir(parents=True, exist_ok=True)
    for stale_path in sdf_dir.glob("*.sdf"):
        stale_path.unlink()
    for stale_path in sdf_dir.glob("*.cif"):
        stale_path.unlink()
    resname_map = {
        residue.label: residue.metadata.get("residue_name", "UNK")
        for residue in PreOrderIter(built.primitive)
        if residue.role == PrimitiveRole.RESIDUE
    }

    # Canonical MuPT export: one RDKit molecule per SEGMENT, with SAAMR hierarchy
    # metadata and PDB-compatible residue metadata supplied by MuPT.
    rdkit_mols = list(primitive_to_rdkit_mols(
        built.primitive,
        resname_map=resname_map,
        default_atom_position=np.zeros(3),
    ))
    # Coordinates were already assigned at the MuPT primitive level from repeat
    # conformers and DPD placement, including sodium positions. Avoid
    # full-chain ETKDG here so production-size systems remain tractable.

    box_parameters = built.primitive.metadata.get("unit_cell_parameters")
    sdf_path = sdf_dir / f"{built.system_name}.sdf"
    writer = Chem.SDWriter(str(sdf_path))
    for mol_idx, mol in enumerate(tqdm(rdkit_mols, desc=f"writing {built.system_name} SDF", unit="mol", leave=False)):
        make_rdkit_molecule_whole(mol, box_parameters)
        prepare_mupt_sdf_atom_props(mol)
        label = segment_label_from_mol(mol, mol_idx)
        if not mol.HasProp("_Name"):
            mol.SetProp("_Name", label)
        writer.write(mol)
    writer.close()
    pdbx_path = sdf_dir / f"{built.system_name}_whole.cif"
    write_rdkit_mols_to_pdbx(rdkit_mols, pdbx_path)
    sdf_paths = [sdf_path]

    built_systems[idx] = BuiltIonomerSystem(
        primitive=built.primitive,
        system_name=built.system_name,
        chain_sequences=built.chain_sequences,
        sodium_count=built.sodium_count,
        sdf_paths=sdf_paths,
    )
    print(
        f"{built.system_name}: wrote {len(rdkit_mols)} molecule record(s) to multi-record SDF {sdf_path}"
    )
    print(f"{built.system_name}: wrote whole-system visualization mmCIF {pdbx_path}")


exporting systems:   0%|          | 0/1 [00:00<?, ?system/s]

writing m5 SDF:   0%|          | 0/1600 [00:00<?, ?mol/s]

m5: wrote 1600 molecule record(s) to multi-record SDF /home/joelaforet/Shirts-Lab-Linux/mupt-examples/examples_system/dpd_role_aware_ionomer_outputs/m5/sdf/m5.sdf


## 7. Fast MuPT SDF Validation

This reload uses MuPT metadata only. It avoids expensive bond and shape reconstruction, which matters for production-size SDF sets.

In [30]:
for built in built_systems:
    records = load_rdkit_sdf_records(built.sdf_paths)
    chain_records = [mol for mol in records if segment_label_from_mol(mol, 0).startswith("chain_")]
    sodium_records = [mol for mol in records if segment_label_from_mol(mol, 0).startswith("sodium_")]
    peaa_residue_ids = {
        (segment_label_from_mol(mol, 0), atom.GetProp("mupt_residue_label"))
        for mol in chain_records
        for atom in mol.GetAtoms()
        if atom.HasProp("mupt_residue_label") and atom.GetProp("mupt_residue_label").endswith("_PEAA")
    }

    assert len(chain_records) == N_CHAINS
    assert len(sodium_records) == built.sodium_count
    assert len(peaa_residue_ids) == built.sodium_count
    print(
        f"{built.system_name}: SDF records chains={len(chain_records)}, "
        f"PEAA={len(peaa_residue_ids)}, sodium={len(sodium_records)}"
    )


m5: SDF records chains=200, PEAA=1400, sodium=1400


## 8. OpenFF/OpenMM Setup

The cells below mirror the companion OpenFF/OpenMM notebook: use OpenFF NAGL GNN charges for polymer chains, avoid AM1-BCC fallback, minimize and briefly collapse in vacuum, then optionally wrap a padded periodic box for NPT. Keep `RUN_OPENFF_PARAMETERIZATION = False` until the smoke-test SDFs look right.

OpenFF can preserve multiple conformers on one `Molecule`, but `Topology.from_molecules(...)` still interprets them as alternate coordinates for the same molecule identity. For this ionomer workflow we therefore read every record from the multi-record SDF and create one OpenFF molecule instance per record.

In [31]:
try:
    from openff.interchange import Interchange
    from openff.toolkit import ForceField, Molecule, Topology
    from openff.toolkit.utils import ToolkitRegistry
    from openff.units import unit as off_unit
    OPENFF_AVAILABLE = True
except ModuleNotFoundError as exc:
    OPENFF_AVAILABLE = False
    OPENFF_IMPORT_ERROR = exc

NAGL_AVAILABLE = False
NAGL_IMPORT_ERROR = None
if OPENFF_AVAILABLE:
    try:
        from openff.toolkit.utils.nagl_wrapper import NAGLToolkitWrapper

        NAGL_AVAILABLE = NAGLToolkitWrapper.is_available()
        if not NAGL_AVAILABLE:
            NAGL_IMPORT_ERROR = RuntimeError("OpenFF NAGL backend is unavailable; install openff-nagl")
    except ModuleNotFoundError as exc:
        NAGL_IMPORT_ERROR = exc

try:
    import openmm
    from openmm import LangevinMiddleIntegrator, MonteCarloBarostat, XmlSerializer
    from openmm import unit as omm_unit
    from openmm.app import DCDReporter, PDBFile, StateDataReporter
    OPENMM_AVAILABLE = True
except ModuleNotFoundError as exc:
    OPENMM_AVAILABLE = False
    OPENMM_IMPORT_ERROR = exc

FORCE_FIELD = "openff-2.2.1.offxml"
PARTIAL_CHARGE_METHOD = "openff-gnn-am1bcc-1.0.0.pt"
RUN_OPENFF_PARAMETERIZATION = True
RUN_VACUUM_COLLAPSE = RUN_OPENFF_PARAMETERIZATION and OPENMM_AVAILABLE
RUN_PERIODIC_NPT = False
DEFAULT_TEMPERATURE_K = 373.0
VACUUM_COLLAPSE_TEMPERATURE_K = 300.0
VACUUM_COLLAPSE_MAX_DURATION_NS = 0.005
VACUUM_COLLAPSE_CHUNK_DURATION_NS = 0.001
VACUUM_COLLAPSE_MIN_CHUNKS = 2
VACUUM_SPAN_RELATIVE_TOLERANCE = 0.03
VACUUM_TIMESTEP_FS = 0.5
VACUUM_N_FRAMES = 50
PERIODIC_INITIAL_DENSITY_G_CM3 = 0.30
PRODUCTION_START_DENSITY_G_CM3 = 0.85
MELT_PREPARATION_PROTOCOL = [
    {"name": "foam collapse", "target_density_g_cm3": 0.35, "temperature_k": 800.0, "pressure_atm": 200.0, "max_duration_ns": 2.0},
    {"name": "dense packing", "target_density_g_cm3": 0.60, "temperature_k": 650.0, "pressure_atm": 100.0, "max_duration_ns": 4.0},
    {"name": "melt approach", "target_density_g_cm3": 0.80, "temperature_k": 500.0, "pressure_atm": 25.0, "max_duration_ns": 6.0},
    {"name": "production-start relaxation", "target_density_g_cm3": PRODUCTION_START_DENSITY_G_CM3, "temperature_k": DEFAULT_TEMPERATURE_K, "pressure_atm": 1.0, "max_duration_ns": 5.0},
]
PERIODIC_NPT_CHUNK_DURATION_NS = 0.02
PERIODIC_TIMESTEP_FS = 2.0
PERIODIC_N_FRAMES = 200
PERIODIC_PADDING_NM = 0.4
MIN_PERIODIC_BOX_CUTOFF_MULTIPLIER = 2.2
OPENMM_PLATFORM_NAME = "CUDA"
OPENMM_PLATFORM_PROPERTIES = {"Precision": "mixed", "DeviceIndex": "0"}
OPENMM_MINIMIZATION_MAX_ITERATIONS = 1_000
OPENMM_RECOVERY_MINIMIZATION_ITERATIONS = 300
OPENMM_STABLE_DYNAMICS_MIN_TIMESTEP_FS = 0.02
OPENMM_STABLE_DYNAMICS_RETRY_SCALE = 0.5
VACUUM_RELAXATION_INITIAL_TEMPERATURE_K = 25.0
VACUUM_RELAXATION_INITIAL_TIMESTEP_FS = 0.05
VACUUM_RELAXATION_RAMP_CHUNKS = 8
OPENMM_REQUIRE_MEMORY_PREFLIGHT = True
OPENMM_MEMORY_ESTIMATE_BYTES_PER_ATOM = 96_000
OPENMM_HOST_MEMORY_SAFETY_FACTOR = 1.5
OPENMM_GPU_MEMORY_SAFETY_FACTOR = 1.2
OPENMM_ALLOW_CPU_FALLBACK_ON_LOW_GPU_MEMORY = False
OPENMM_CONTACT_DETAIL_LIMIT = 12
OPENMM_CONTACT_DISTANCE_TOLERANCE_A = 1.0e-3

print(f"OpenFF available: {OPENFF_AVAILABLE}")
if not OPENFF_AVAILABLE:
    print(f"  {OPENFF_IMPORT_ERROR}")
print(f"OpenFF NAGL available: {NAGL_AVAILABLE}")
if not NAGL_AVAILABLE and NAGL_IMPORT_ERROR is not None:
    print(f"  {NAGL_IMPORT_ERROR}")
print(f"OpenMM available: {OPENMM_AVAILABLE}")
if not OPENMM_AVAILABLE:
    print(f"  {OPENMM_IMPORT_ERROR}")
print(f"Run OpenFF parameterization: {RUN_OPENFF_PARAMETERIZATION}")


def openmm_platform_kwargs() -> dict:
    """Prefer CUDA for OpenMM when available, otherwise let OpenMM choose."""
    if not OPENMM_AVAILABLE:
        return {}
    try:
        platform = openmm.Platform.getPlatformByName(OPENMM_PLATFORM_NAME)
    except Exception as exc:
        print(f"OpenMM {OPENMM_PLATFORM_NAME} platform unavailable; using OpenMM default platform ({exc})")
        return {}
    return {"platform": platform, "platformProperties": OPENMM_PLATFORM_PROPERTIES}


def bytes_to_gib(value: int | float | None) -> float | None:
    if value is None:
        return None
    return float(value) / 1024.0**3


def host_memory_bytes() -> dict[str, int | None]:
    """Return total and available host memory from /proc/meminfo when available."""
    meminfo_path = Path("/proc/meminfo")
    if not meminfo_path.exists():
        return {"total": None, "available": None}
    values = {}
    for line in meminfo_path.read_text().splitlines():
        key, raw_value = line.split(":", 1)
        fields = raw_value.strip().split()
        if fields:
            values[key] = int(fields[0]) * 1024
    return {"total": values.get("MemTotal"), "available": values.get("MemAvailable")}


def gpu_memory_bytes(device_index: str = "0") -> dict[str, int | None]:
    """Return total and free GPU memory using nvidia-smi when available."""
    if shutil.which("nvidia-smi") is None:
        return {"total": None, "free": None}
    try:
        result = subprocess.run(
            [
                "nvidia-smi",
                f"--id={device_index}",
                "--query-gpu=memory.total,memory.free",
                "--format=csv,noheader,nounits",
            ],
            check=True,
            capture_output=True,
            text=True,
        )
    except Exception:
        return {"total": None, "free": None}
    first_line = result.stdout.strip().splitlines()[0]
    total_mib, free_mib = [int(value.strip()) for value in first_line.split(",")[:2]]
    return {"total": total_mib * 1024**2, "free": free_mib * 1024**2}


def openmm_memory_preflight(atom_count: int, label: str, platform_name: str) -> dict[str, float | int | bool | str | None]:
    """Estimate whether the local machine can create the OpenMM context."""
    estimate_bytes = int(atom_count * OPENMM_MEMORY_ESTIMATE_BYTES_PER_ATOM)
    host = host_memory_bytes()
    device_index = str(OPENMM_PLATFORM_PROPERTIES.get("DeviceIndex", "0")).split(",")[0]
    gpu = gpu_memory_bytes(device_index) if platform_name.upper() in {"CUDA", "OPENCL"} else {"total": None, "free": None}
    required_host = int(estimate_bytes * OPENMM_HOST_MEMORY_SAFETY_FACTOR)
    required_gpu = int(estimate_bytes * OPENMM_GPU_MEMORY_SAFETY_FACTOR)
    host_ok = host["available"] is None or host["available"] >= required_host
    gpu_ok = gpu["free"] is None or gpu["free"] >= required_gpu
    summary = {
        "label": label,
        "atom_count": int(atom_count),
        "estimate_gib": bytes_to_gib(estimate_bytes),
        "required_host_gib": bytes_to_gib(required_host),
        "host_total_gib": bytes_to_gib(host["total"]),
        "host_available_gib": bytes_to_gib(host["available"]),
        "required_gpu_gib": bytes_to_gib(required_gpu),
        "gpu_total_gib": bytes_to_gib(gpu["total"]),
        "gpu_free_gib": bytes_to_gib(gpu["free"]),
        "host_ok": bool(host_ok),
        "gpu_ok": bool(gpu_ok),
    }

    def fmt(value):
        return "unknown" if value is None else f"{value:.1f} GiB"

    print(
        f"{label}: OpenMM memory preflight atoms={atom_count:,}, estimated working set={fmt(summary['estimate_gib'])}, "
        f"host available={fmt(summary['host_available_gib'])} / required={fmt(summary['required_host_gib'])}, "
        f"GPU free={fmt(summary['gpu_free_gib'])} / required={fmt(summary['required_gpu_gib'])}"
    )
    if OPENMM_REQUIRE_MEMORY_PREFLIGHT and (not host_ok or not gpu_ok):
        raise MemoryError(
            f"{label}: local memory preflight failed before OpenMM context creation. "
            f"Summary: {summary}. Reduce N_CHAINS, use a larger-memory GPU/node, or set "
            "OPENMM_REQUIRE_MEMORY_PREFLIGHT = False to attempt the run anyway."
        )
    return summary


OpenFF available: True
OpenFF NAGL available: True
OpenMM available: True
Run OpenFF parameterization: True


In [32]:
def transfer_rdkit_metadata_to_openff(rdkit_mol: Chem.Mol, off_mol) -> None:
    """Copy SDF atom-property metadata into OpenFF atom metadata."""
    for rd_atom, off_atom in zip(rdkit_mol.GetAtoms(), off_mol.atoms):
        props = rd_atom.GetPropsAsDict(includePrivate=True, includeComputed=False)
        off_atom.metadata.update({
            "residue_name": str(props.get("residue_name", "UNK")),
            "residue_number": str(props.get("residue_id", props.get("mupt_residue_index", "1"))),
            "chain_id": str(props.get("chain_id", "A")),
            "atom_name": f"{rd_atom.GetSymbol()}{rd_atom.GetIdx() + 1}",
        })


def load_rdkit_sdf_records(paths: list[Path]) -> list[Chem.Mol]:
    """Load every SDF molecule record without stripping explicit hydrogens."""
    records = []
    for path in paths:
        supplier = Chem.SDMolSupplier(str(path), removeHs=False, sanitize=False)
        for record_idx, mol in enumerate(supplier):
            if mol is None:
                raise ValueError(f"Could not read record {record_idx} from {path}")
            sanitized = Chem.Mol(mol)
            Chem.SanitizeMol(sanitized)
            records.append(sanitized)
    return records


openff_molecules_by_system = {}
if OPENFF_AVAILABLE:
    for built in built_systems:
        off_molecules = []
        for rdkit_mol in load_rdkit_sdf_records(built.sdf_paths):
            off_mol = Molecule.from_rdkit(
                rdkit_mol,
                allow_undefined_stereo=True,
                hydrogens_are_explicit=True,
            )
            transfer_rdkit_metadata_to_openff(rdkit_mol, off_mol)
            off_molecules.append(off_mol)
        openff_molecules_by_system[built.system_name] = off_molecules
        print(f"{built.system_name}: created {len(off_molecules)} OpenFF molecule(s)")
else:
    print("Skipping OpenFF molecule conversion because openff-toolkit is unavailable.")

m5: created 1600 OpenFF molecule(s)


In [33]:
def is_sodium_molecule(off_mol: Molecule) -> bool:
    """Return True for the one-atom sodium counterion molecules."""
    return off_mol.n_atoms == 1 and off_mol.atom(0).symbol == "Na"


def representative_charge_molecules(off_molecules: list[Molecule], system_name: str, nagl_registry: ToolkitRegistry) -> list[Molecule]:
    """Return one charged representative for each expected molecule chemistry."""
    chain_mol = next((mol for mol in off_molecules if not is_sodium_molecule(mol)), None)
    sodium_mol = next((mol for mol in off_molecules if is_sodium_molecule(mol)), None)
    if chain_mol is None:
        raise ValueError(f"{system_name}: no polymer chain molecule found for charge assignment")

    charge_start_s = time.perf_counter()
    print(f"{system_name}: assigning NAGL charges for representative polymer chain ({chain_mol.n_atoms} atoms)")
    chain_mol.assign_partial_charges(
        partial_charge_method=PARTIAL_CHARGE_METHOD,
        toolkit_registry=nagl_registry,
    )
    print(f"{system_name}: representative polymer charge assignment took {time.perf_counter() - charge_start_s:.2f} s")
    charge_molecules = [chain_mol]

    if sodium_mol is not None:
        print(f"{system_name}: assigning sodium charges for representative Na+ molecule")
        sodium_mol.partial_charges = [1.0] * off_unit.elementary_charge
        charge_molecules.append(sodium_mol)

    return charge_molecules


def representative_topology_and_positions(off_molecules: list[Molecule], system_name: str):
    """Build an OpenFF topology from repeated templates and preserve instance positions."""
    chain_template = next((mol for mol in off_molecules if not is_sodium_molecule(mol)), None)
    sodium_template = next((mol for mol in off_molecules if is_sodium_molecule(mol)), None)
    if chain_template is None:
        raise ValueError(f"{system_name}: no polymer chain molecule found for topology construction")

    topology_molecules = []
    positions = []
    for mol in off_molecules:
        template = sodium_template if is_sodium_molecule(mol) else chain_template
        if template is None or mol.n_atoms != template.n_atoms:
            raise ValueError(
                f"{system_name}: molecule atom count {mol.n_atoms} does not match its representative template"
            )
        if not mol.conformers:
            raise ValueError(f"{system_name}: molecule is missing SDF coordinates")
        topology_molecules.append(template)
        positions.append(mol.conformers[0].m_as(off_unit.angstrom))

    topology = Topology.from_molecules(topology_molecules)
    return topology, np.vstack(positions) * off_unit.angstrom


interchanges_by_system = {}

if OPENFF_AVAILABLE and NAGL_AVAILABLE and RUN_OPENFF_PARAMETERIZATION:
    ff = ForceField(FORCE_FIELD)
    nagl_registry = ToolkitRegistry([NAGLToolkitWrapper()])

    for built in built_systems:
        off_molecules = openff_molecules_by_system[built.system_name]
        cell_start_s = time.perf_counter()
        print(f"{built.system_name}: building representative OpenFF topology from {len(off_molecules)} molecule record(s)")
        topology_start_s = time.perf_counter()
        topology, topology_positions = representative_topology_and_positions(off_molecules, built.system_name)
        print(f"{built.system_name}: OpenFF topology construction took {time.perf_counter() - topology_start_s:.2f} s")
        unique_charge_molecules = representative_charge_molecules(
            off_molecules,
            system_name=built.system_name,
            nagl_registry=nagl_registry,
        )
        print(
            f"{built.system_name}: creating Interchange using {len(unique_charge_molecules)} "
            f"representative charged molecule type(s) across {len(off_molecules)} total molecules"
        )

        interchange_start_s = time.perf_counter()
        interchange = ff.create_interchange(
            topology,
            charge_from_molecules=unique_charge_molecules,
        )
        interchange.positions = topology_positions
        box_parameters = built.primitive.metadata.get("unit_cell_parameters")
        if box_parameters is None:
            raise ValueError(f"{built.system_name}: missing DPD unit_cell_parameters for periodic OpenMM setup")
        box_lengths_nm = np.asarray(box_parameters[:3], dtype=float) * 0.1
        interchange.box = np.diag(box_lengths_nm) * off_unit.nanometer
        print(f"{built.system_name}: Interchange creation took {time.perf_counter() - interchange_start_s:.2f} s")
        print(f"{built.system_name}: using DPD periodic box lengths {box_lengths_nm[0]:.2f} x {box_lengths_nm[1]:.2f} x {box_lengths_nm[2]:.2f} nm")
        interchanges_by_system[built.system_name] = interchange
        print(
            f"{built.system_name}: created vacuum interchange with {interchange.topology.n_atoms} atoms "
            f"from {len(unique_charge_molecules)} unique molecule type(s) in {time.perf_counter() - cell_start_s:.2f} s"
        )
elif OPENFF_AVAILABLE and not NAGL_AVAILABLE:
    print("Parameterization skipped because OpenFF NAGL is unavailable; no AM1-BCC fallback is used.")
else:
    print("Parameterization skipped. Set RUN_OPENFF_PARAMETERIZATION = True after checking the smoke-test SDFs.")

m5: building representative OpenFF topology from 1600 molecule record(s)
m5: OpenFF topology construction took 7.64 s
m5: assigning NAGL charges for representative polymer chain (352 atoms)
m5: representative polymer charge assignment took 1.65 s
m5: assigning sodium charges for representative Na+ molecule
m5: creating Interchange using 2 representative charged molecule type(s) across 1600 total molecules
m5: Interchange creation took 37.23 s
m5: created vacuum interchange with 71800 atoms from 2 unique molecule type(s) in 46.53 s


## 9. Vacuum Collapse and Optional Periodic NPT

This follows the companion notebook's pragmatic route: collapse in vacuum first to avoid building an enormous sparse PME grid around initial coordinates. Only enable periodic NPT after inspecting the vacuum result.

In [ ]:
DA_PER_NM3_TO_G_CM3 = 1.0 / 602.214076


def simulation_steps(duration_ns: float, timestep_fs: float) -> int:
    """Convert a duration in ns and timestep in fs to OpenMM steps."""
    if duration_ns < 0:
        raise ValueError("duration_ns must be non-negative")
    if timestep_fs <= 0:
        raise ValueError("timestep_fs must be positive")
    return int(round(duration_ns * 1_000_000.0 / timestep_fs))


def report_interval(total_steps: int, n_frames: int) -> int:
    """Return a reporter interval that saves approximately n_frames frames."""
    if n_frames <= 0:
        raise ValueError("n_frames must be positive")
    return max(1, total_steps // n_frames) if total_steps else 1


def openmm_system_mass_da(system) -> float:
    """Return total OpenMM system mass in daltons."""
    return sum(system.getParticleMass(i).value_in_unit(omm_unit.dalton) for i in range(system.getNumParticles()))


def maximum_nonbonded_cutoff_nm(system) -> float:
    """Return the largest cutoff distance used by OpenMM nonbonded-like forces."""
    cutoffs = []
    for force in system.getForces():
        if hasattr(force, "getCutoffDistance"):
            try:
                cutoffs.append(force.getCutoffDistance().value_in_unit(omm_unit.nanometer))
            except Exception:
                pass
    if not cutoffs:
        raise ValueError("Could not determine a nonbonded cutoff from the OpenMM system")
    return float(max(cutoffs))


def density_from_box_g_cm3(mass_da: float, box_vectors_nm: np.ndarray) -> float:
    """Return mass density from box vectors in nm."""
    volume_nm3 = abs(float(np.linalg.det(box_vectors_nm)))
    return mass_da * DA_PER_NM3_TO_G_CM3 / volume_nm3


def coordinate_span_nm(positions_nm: np.ndarray) -> np.ndarray:
    """Return orthorhombic coordinate span in nm."""
    return positions_nm.max(axis=0) - positions_nm.min(axis=0)


def box_lengths_nm(box_vectors_nm: np.ndarray) -> np.ndarray:
    """Return periodic box vector lengths in nm."""
    return np.array([np.linalg.norm(vector) for vector in box_vectors_nm], dtype=float)


def set_barostat_conditions(simulation, temperature_k: float, pressure_atm: float) -> None:
    """Update integrator and barostat thermodynamic targets in-place."""
    simulation.integrator.setTemperature(temperature_k * omm_unit.kelvin)
    pressure_bar = (pressure_atm * omm_unit.atmosphere).value_in_unit(omm_unit.bar)
    simulation.context.setParameter(MonteCarloBarostat.Temperature(), temperature_k)
    simulation.context.setParameter(MonteCarloBarostat.Pressure(), pressure_bar)


def set_periodic_box_for_initial_density(interchange, mass_da: float, density_g_cm3: float, padding_nm: float):
    """Set the smallest orthorhombic box satisfying density and padding constraints."""
    positions_nm = interchange.positions.m_as(off_unit.nanometer)
    mins = positions_nm.min(axis=0)
    maxs = positions_nm.max(axis=0)
    shifted_positions = positions_nm - mins + padding_nm
    span_lengths = (maxs - mins) + 2 * padding_nm
    target_length = (mass_da * DA_PER_NM3_TO_G_CM3 / density_g_cm3) ** (1.0 / 3.0)
    box_lengths = np.maximum(span_lengths, target_length)
    interchange.positions = shifted_positions * off_unit.nanometer
    interchange.box = np.diag(box_lengths) * off_unit.nanometer
    return box_lengths



def metadata_openmm_topology(off_molecules, box_vectors_nm: np.ndarray | None = None) -> openmm.app.Topology:
    """Build an OpenMM topology grouped by MuPT/PDB atom metadata."""
    topology = openmm.app.Topology()
    chains = {}
    residues = {}
    atom_lookup = {}

    for mol_idx, off_mol in enumerate(off_molecules):
        for atom_idx, off_atom in enumerate(off_mol.atoms):
            chain_id = str(off_atom.metadata.get("chain_id", "A"))
            residue_number = str(off_atom.metadata.get("residue_number", "1"))
            residue_name = str(off_atom.metadata.get("residue_name", "UNK"))
            atom_name = str(off_atom.metadata.get("atom_name", f"{off_atom.symbol}{atom_idx + 1}"))

            chain = chains.get(chain_id)
            if chain is None:
                chain = topology.addChain(chain_id)
                chains[chain_id] = chain

            residue_key = (chain_id, residue_number, residue_name)
            residue = residues.get(residue_key)
            if residue is None:
                residue = topology.addResidue(residue_name, chain, id=residue_number)
                residues[residue_key] = residue

            element = openmm.app.element.get_by_symbol(off_atom.symbol)
            atom_lookup[(mol_idx, atom_idx)] = topology.addAtom(atom_name, element, residue)

        for bond in off_mol.bonds:
            topology.addBond(
                atom_lookup[(mol_idx, bond.atom1_index)],
                atom_lookup[(mol_idx, bond.atom2_index)],
            )

    if box_vectors_nm is not None:
        topology.setPeriodicBoxVectors(box_vectors_nm * omm_unit.nanometer)

    return topology


def write_production_start_files(
    simulation,
    off_molecules,
    output_dir: Path,
    system_name: str,
    system_mass_da: float,
    steps_run: int,
    timestep_fs: float,
    temperature_k: float,
    pressure_atm: float,
    density_g_cm3: float,
) -> dict[str, Path]:
    """Write restart-ready coordinates, box vectors, and metadata for production MD."""
    production_dir = output_dir / "production_start"
    production_dir.mkdir(parents=True, exist_ok=True)

    state = simulation.context.getState(
        getEnergy=True,
        getPositions=True,
        getVelocities=True,
        enforcePeriodicBox=True,
    )
    positions_nm = state.getPositions(asNumpy=True).value_in_unit(omm_unit.nanometer)
    box_vectors_nm = state.getPeriodicBoxVectors(asNumpy=True).value_in_unit(omm_unit.nanometer)
    density = density_from_box_g_cm3(system_mass_da, box_vectors_nm)
    lengths_nm = box_lengths_nm(box_vectors_nm)

    topology = metadata_openmm_topology(off_molecules, box_vectors_nm=box_vectors_nm)
    pdb_path = production_dir / f"{system_name}_production_start.pdb"
    state_path = production_dir / f"{system_name}_production_start_state.xml"
    checkpoint_path = production_dir / f"{system_name}_production_start.chk"
    arrays_path = production_dir / f"{system_name}_production_start_arrays.npz"
    manifest_path = production_dir / f"{system_name}_production_start_manifest.json"

    with pdb_path.open("w") as handle:
        PDBFile.writeFile(topology, state.getPositions(asNumpy=True), handle)
    state_path.write_text(XmlSerializer.serialize(state))
    simulation.saveCheckpoint(str(checkpoint_path))
    np.savez_compressed(
        arrays_path,
        positions_nm=positions_nm,
        box_vectors_nm=box_vectors_nm,
        box_lengths_nm=lengths_nm,
    )

    manifest = {
        "system_name": system_name,
        "intended_use": "starting coordinates for subsequent production MD",
        "steps_run": int(steps_run),
        "elapsed_ns": float(steps_run * timestep_fs / 1_000_000.0),
        "temperature_k": float(temperature_k),
        "pressure_atm": float(pressure_atm),
        "target_density_g_cm3": float(density_g_cm3),
        "final_density_g_cm3": float(density),
        "mass_da": float(system_mass_da),
        "box_lengths_nm": [float(x) for x in lengths_nm],
        "box_vectors_nm": box_vectors_nm.tolist(),
        "files": {
            "pdb": str(pdb_path.relative_to(EXAMPLES_ROOT)),
            "state_xml": str(state_path.relative_to(EXAMPLES_ROOT)),
            "checkpoint": str(checkpoint_path.relative_to(EXAMPLES_ROOT)),
            "arrays_npz": str(arrays_path.relative_to(EXAMPLES_ROOT)),
        },
    }
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")

    return {
        "pdb": pdb_path,
        "state_xml": state_path,
        "checkpoint": checkpoint_path,
        "arrays_npz": arrays_path,
        "manifest": manifest_path,
    }


def attach_openmm_reporters(simulation, trajectory_path: Path, state_data_path: Path, interval: int, include_density: bool) -> None:
    """Attach trajectory and state-data reporters for later analysis."""
    simulation.reporters.append(DCDReporter(str(trajectory_path), interval))
    simulation.reporters.append(
        StateDataReporter(
            str(state_data_path),
            reportInterval=interval,
            step=True,
            time=True,
            potentialEnergy=True,
            kineticEnergy=True,
            temperature=True,
            volume=include_density,
            density=include_density,
            speed=True,
        )
    )


def finite_openmm_positions(simulation) -> bool:
    """Return False when OpenMM positions contain NaN or inf values."""
    state = simulation.context.getState(getPositions=True)
    positions_nm = state.getPositions(asNumpy=True).value_in_unit(omm_unit.nanometer)
    return bool(np.all(np.isfinite(positions_nm)))


def run_recoverable_openmm_chunk(
    simulation,
    steps: int,
    timestep_fs: float,
    label: str,
) -> float:
    """Run one dynamics chunk, restoring a checkpoint and shrinking dt on NaNs."""
    if steps <= 0:
        return timestep_fs
    trial_timestep_fs = float(timestep_fs)
    checkpoint = simulation.context.createCheckpoint()
    while trial_timestep_fs >= OPENMM_STABLE_DYNAMICS_MIN_TIMESTEP_FS:
        simulation.integrator.setStepSize(trial_timestep_fs * omm_unit.femtosecond)
        try:
            simulation.step(steps)
            if not finite_openmm_positions(simulation):
                raise RuntimeError("OpenMM positions became non-finite")
            return trial_timestep_fs
        except Exception as exc:
            print(
                f"{label}: dynamics failed at dt={trial_timestep_fs:.4f} fs ({exc}); "
                "restoring checkpoint, minimizing, and retrying with a smaller timestep"
            )
            simulation.context.loadCheckpoint(checkpoint)
            simulation.minimizeEnergy(maxIterations=OPENMM_RECOVERY_MINIMIZATION_ITERATIONS)
            checkpoint = simulation.context.createCheckpoint()
            trial_timestep_fs *= OPENMM_STABLE_DYNAMICS_RETRY_SCALE
    raise RuntimeError(
        f"{label}: dynamics remained unstable below {OPENMM_STABLE_DYNAMICS_MIN_TIMESTEP_FS} fs; "
        "inspect contacts or force-field assignment before continuing"
    )


def summarize_periodic_contact_clashes(off_molecules, box_lengths_nm: np.ndarray, cutoff_a: float = 2.0) -> dict[str, int | float | list[dict]]:
    """Count non-excluded periodic atom contacts below a short-distance cutoff."""
    box_lengths_a = np.asarray(box_lengths_nm, dtype=float) * 10.0
    positions = []
    atom_symbols = []
    atom_metadata = []
    atom_molecule_indices = []
    bonds = []
    offset = 0

    for mol_idx, off_mol in enumerate(off_molecules):
        molecule_positions = off_mol.conformers[0].m_as(off_unit.angstrom)
        positions.extend(molecule_positions)
        atom_symbols.extend(atom.symbol for atom in off_mol.atoms)
        atom_metadata.extend(dict(atom.metadata) for atom in off_mol.atoms)
        atom_molecule_indices.extend([mol_idx] * off_mol.n_atoms)
        for bond in off_mol.bonds:
            bonds.append((offset + bond.atom1_index, offset + bond.atom2_index))
        offset += off_mol.n_atoms

    positions = np.asarray(positions, dtype=float) % box_lengths_a
    adjacency = [set() for _ in range(len(positions))]
    for atom_1, atom_2 in bonds:
        adjacency[atom_1].add(atom_2)
        adjacency[atom_2].add(atom_1)

    excluded_pairs = set()
    for atom_idx, neighbors in enumerate(adjacency):
        for neighbor_idx in neighbors:
            excluded_pairs.add(tuple(sorted((atom_idx, neighbor_idx))))
        for neighbor_idx in neighbors:
            for next_idx in adjacency[neighbor_idx]:
                if next_idx != atom_idx:
                    excluded_pairs.add(tuple(sorted((atom_idx, next_idx))))
        frontier = set(neighbors)
        visited = {atom_idx}
        for _ in range(3):
            next_frontier = set()
            for neighbor_idx in frontier:
                if neighbor_idx == atom_idx:
                    continue
                excluded_pairs.add(tuple(sorted((atom_idx, neighbor_idx))))
                next_frontier.update(adjacency[neighbor_idx] - visited)
            visited.update(frontier)
            frontier = next_frontier

    tree = cKDTree(positions, boxsize=box_lengths_a)
    raw_pairs = tree.query_pairs(cutoff_a, output_type="ndarray")
    nonexcluded = 0
    sodium_contacts = 0
    intramolecular = 0
    intermolecular = 0
    min_distance = np.inf
    contact_details = []
    for atom_1, atom_2 in raw_pairs:
        atom_1 = int(atom_1)
        atom_2 = int(atom_2)
        if tuple(sorted((atom_1, atom_2))) in excluded_pairs:
            continue
        delta = positions[atom_2] - positions[atom_1]
        delta -= np.round(delta / box_lengths_a) * box_lengths_a
        distance = float(np.linalg.norm(delta))
        if distance >= cutoff_a - OPENMM_CONTACT_DISTANCE_TOLERANCE_A:
            continue
        nonexcluded += 1
        if atom_molecule_indices[atom_1] == atom_molecule_indices[atom_2]:
            intramolecular += 1
        else:
            intermolecular += 1
        if atom_symbols[atom_1] == "Na" or atom_symbols[atom_2] == "Na":
            sodium_contacts += 1
        min_distance = min(min_distance, distance)
        if len(contact_details) < OPENMM_CONTACT_DETAIL_LIMIT or distance < contact_details[-1]["distance_a"]:
            contact_details.append({
                "distance_a": distance,
                "sodium_contact": atom_symbols[atom_1] == "Na" or atom_symbols[atom_2] == "Na",
                "intramolecular": atom_molecule_indices[atom_1] == atom_molecule_indices[atom_2],
                "atom_1": atom_metadata[atom_1].get("atom_name", atom_symbols[atom_1]),
                "atom_2": atom_metadata[atom_2].get("atom_name", atom_symbols[atom_2]),
                "residue_1": atom_metadata[atom_1].get("residue_name", "UNK"),
                "residue_2": atom_metadata[atom_2].get("residue_name", "UNK"),
                "chain_1": atom_metadata[atom_1].get("chain_id", "?"),
                "chain_2": atom_metadata[atom_2].get("chain_id", "?"),
                "residue_number_1": atom_metadata[atom_1].get("residue_number", "?"),
                "residue_number_2": atom_metadata[atom_2].get("residue_number", "?"),
            })
            contact_details = sorted(contact_details, key=lambda item: item["distance_a"])[:OPENMM_CONTACT_DETAIL_LIMIT]

    if not np.isfinite(min_distance):
        min_distance = np.nan
    return {
        "cutoff_a": float(cutoff_a),
        "raw_pairs": int(len(raw_pairs)),
        "nonexcluded": int(nonexcluded),
        "sodium_contacts": int(sodium_contacts),
        "intramolecular": int(intramolecular),
        "intermolecular": int(intermolecular),
        "min_distance_a": float(min_distance),
        "details": contact_details,
    }


if interchanges_by_system and OPENMM_AVAILABLE and RUN_VACUUM_COLLAPSE:
    vacuum_max_steps = simulation_steps(VACUUM_COLLAPSE_MAX_DURATION_NS, VACUUM_TIMESTEP_FS)
    vacuum_chunk_steps = max(1, simulation_steps(VACUUM_COLLAPSE_CHUNK_DURATION_NS, VACUUM_TIMESTEP_FS))
    vacuum_interval = report_interval(vacuum_max_steps, VACUUM_N_FRAMES)
    periodic_chunk_steps = max(1, simulation_steps(PERIODIC_NPT_CHUNK_DURATION_NS, PERIODIC_TIMESTEP_FS))
    periodic_max_steps = sum(
        simulation_steps(stage["max_duration_ns"], PERIODIC_TIMESTEP_FS)
        for stage in MELT_PREPARATION_PROTOCOL
    )
    periodic_interval = report_interval(periodic_max_steps, PERIODIC_N_FRAMES)

    for built in built_systems:
        interchange = interchanges_by_system[built.system_name]
        openmm_dir = OUTPUT_ROOT / built.system_name / "OpenMM"
        openmm_dir.mkdir(parents=True, exist_ok=True)

        first_stage = MELT_PREPARATION_PROTOCOL[0]
        pressure = first_stage["pressure_atm"] * omm_unit.atmosphere
        vacuum_temperature = VACUUM_COLLAPSE_TEMPERATURE_K * omm_unit.kelvin
        initial_vacuum_temperature = VACUUM_RELAXATION_INITIAL_TEMPERATURE_K * omm_unit.kelvin
        periodic_temperature = first_stage["temperature_k"] * omm_unit.kelvin
        vacuum_time_step = VACUUM_RELAXATION_INITIAL_TIMESTEP_FS * omm_unit.femtosecond
        periodic_time_step = PERIODIC_TIMESTEP_FS * omm_unit.femtosecond
        friction = 10.0 / omm_unit.picosecond
        system_name = f"ionomer_{built.system_name}"

        initial_box_lengths_nm = box_lengths_nm(interchange.box.m_as(off_unit.nanometer))
        clash_summary = summarize_periodic_contact_clashes(openff_molecules_by_system[built.system_name], initial_box_lengths_nm)
        print(f"{built.system_name}: periodic contact preflight under {clash_summary['cutoff_a']:.1f} A = {clash_summary}")
        severe_contact = np.isfinite(clash_summary["min_distance_a"]) and clash_summary["min_distance_a"] < 1.5
        if clash_summary["sodium_contacts"] > 0 or severe_contact:
            raise RuntimeError(
                f"{built.system_name}: periodic contact preflight found severe contacts before OpenMM dynamics: "
                f"{clash_summary}. Rerun cells 4-8 after the periodic clash-aware sodium placement patch, "
                "then rerun cell 9."
            )
        if clash_summary["nonexcluded"] > 0:
            print(
                f"{built.system_name}: warning: {clash_summary['nonexcluded']} mild non-excluded contacts below "
                f"{clash_summary['cutoff_a']:.1f} A remain, but none involve sodium and the shortest is "
                f"{clash_summary['min_distance_a']:.2f} A; proceeding to minimization."
            )

        print(f"{built.system_name}: creating periodic OpenMM simulation from DPD box")
        openmm_memory_preflight(interchange.topology.n_atoms, f"{built.system_name}: initial OpenMM context", OPENMM_PLATFORM_NAME)
        vacuum_integrator = LangevinMiddleIntegrator(initial_vacuum_temperature, friction, vacuum_time_step)
        vacuum_simulation = interchange.to_openmm_simulation(
            integrator=vacuum_integrator,
            combine_nonbonded_forces=False,
            **openmm_platform_kwargs(),
        )
        print(f"{built.system_name}: OpenMM platform = {vacuum_simulation.context.getPlatform().getName()}")
        system_mass_da = openmm_system_mass_da(vacuum_simulation.system)
        print(f"{built.system_name}: running periodic minimization for up to {OPENMM_MINIMIZATION_MAX_ITERATIONS} iterations")
        vacuum_simulation.minimizeEnergy(maxIterations=OPENMM_MINIMIZATION_MAX_ITERATIONS)
        vacuum_simulation.context.setVelocitiesToTemperature(initial_vacuum_temperature, RANDOM_SEED)
        if vacuum_max_steps > 0:
            vacuum_dcd_path = openmm_dir / f"{system_name}_vacuum_trajectory.dcd"
            vacuum_state_data_path = openmm_dir / f"{system_name}_vacuum_state_data.csv"
            attach_openmm_reporters(vacuum_simulation, vacuum_dcd_path, vacuum_state_data_path, vacuum_interval, include_density=False)
            print(
                f"{built.system_name}: running periodic high-temperature relaxation until coordinate span stabilizes "
                f"or {VACUUM_COLLAPSE_MAX_DURATION_NS} ns at {VACUUM_COLLAPSE_TEMPERATURE_K} K"
            )
            vacuum_steps_run = 0
            vacuum_elapsed_ns = 0.0
            current_vacuum_timestep_fs = VACUUM_RELAXATION_INITIAL_TIMESTEP_FS
            previous_span_volume = None
            vacuum_chunk_idx = 0
            while vacuum_steps_run < vacuum_max_steps:
                steps_this_chunk = min(vacuum_chunk_steps, vacuum_max_steps - vacuum_steps_run)
                ramp_fraction = min(1.0, (vacuum_chunk_idx + 1) / max(1, VACUUM_RELAXATION_RAMP_CHUNKS))
                target_temperature_k = (
                    VACUUM_RELAXATION_INITIAL_TEMPERATURE_K
                    + ramp_fraction * (VACUUM_COLLAPSE_TEMPERATURE_K - VACUUM_RELAXATION_INITIAL_TEMPERATURE_K)
                )
                target_timestep_fs = (
                    VACUUM_RELAXATION_INITIAL_TIMESTEP_FS
                    + ramp_fraction * (VACUUM_TIMESTEP_FS - VACUUM_RELAXATION_INITIAL_TIMESTEP_FS)
                )
                current_vacuum_timestep_fs = min(target_timestep_fs, max(current_vacuum_timestep_fs * 1.25, OPENMM_STABLE_DYNAMICS_MIN_TIMESTEP_FS))
                vacuum_integrator.setTemperature(target_temperature_k * omm_unit.kelvin)
                current_vacuum_timestep_fs = run_recoverable_openmm_chunk(
                    vacuum_simulation,
                    steps_this_chunk,
                    current_vacuum_timestep_fs,
                    f"{built.system_name}: vacuum relaxation chunk {vacuum_chunk_idx + 1}",
                )
                vacuum_steps_run += steps_this_chunk
                vacuum_elapsed_ns += steps_this_chunk * current_vacuum_timestep_fs / 1_000_000.0
                vacuum_chunk_idx += 1
                state = vacuum_simulation.context.getState(getPositions=True)
                positions_nm = state.getPositions(asNumpy=True).value_in_unit(omm_unit.nanometer)
                span = coordinate_span_nm(positions_nm)
                span_volume = float(np.prod(span))
                relative_change = np.inf if previous_span_volume is None else abs(span_volume - previous_span_volume) / previous_span_volume
                print(
                    f"{built.system_name}: vacuum {vacuum_elapsed_ns:.4f} ns, "
                    f"T={target_temperature_k:.1f} K, dt={current_vacuum_timestep_fs:.4f} fs, "
                    f"span={span[0]:.2f} x {span[1]:.2f} x {span[2]:.2f} nm, "
                    f"span-volume change={relative_change:.3f}"
                )
                if vacuum_chunk_idx >= VACUUM_COLLAPSE_MIN_CHUNKS and relative_change < VACUUM_SPAN_RELATIVE_TOLERANCE:
                    print(f"{built.system_name}: vacuum collapse converged by span-volume criterion")
                    break
                previous_span_volume = span_volume
            print(f"{built.system_name}: vacuum trajectory saved to {vacuum_dcd_path.relative_to(EXAMPLES_ROOT)}")
            print(f"{built.system_name}: vacuum state data saved to {vacuum_state_data_path.relative_to(EXAMPLES_ROOT)}")

        vacuum_state = vacuum_simulation.context.getState(getPositions=True, getEnergy=True)
        collapsed_positions_nm = vacuum_state.getPositions(asNumpy=True).value_in_unit(omm_unit.nanometer)
        interchange.positions = collapsed_positions_nm * off_unit.nanometer

        vacuum_topology_path = openmm_dir / f"{system_name}_vacuum_topology.pdb"
        vacuum_system_path = openmm_dir / f"{system_name}_vacuum_system.xml"
        vacuum_integrator_path = openmm_dir / f"{system_name}_vacuum_integrator.xml"
        vacuum_state_path = openmm_dir / f"{system_name}_vacuum_state.xml"
        with vacuum_topology_path.open("w") as handle:
            PDBFile.writeFile(metadata_openmm_topology(openff_molecules_by_system[built.system_name]), vacuum_state.getPositions(asNumpy=True), handle)
        vacuum_system_path.write_text(XmlSerializer.serialize(vacuum_simulation.system))
        vacuum_integrator_path.write_text(XmlSerializer.serialize(vacuum_integrator))
        vacuum_state_path.write_text(XmlSerializer.serialize(vacuum_state))

        print(f"{built.system_name}: vacuum potential energy: {vacuum_state.getPotentialEnergy()}")
        print(f"{built.system_name}: serialized vacuum OpenMM components:")
        for output_path in (vacuum_topology_path, vacuum_system_path, vacuum_integrator_path, vacuum_state_path):
            print(f"  {output_path.relative_to(EXAMPLES_ROOT)}")

        if RUN_PERIODIC_NPT:
            box_lengths = box_lengths_nm(interchange.box.m_as(off_unit.nanometer))
            print(
                f"{built.system_name}: starting periodic NPT from box lengths "
                f"{box_lengths[0]:.2f} x {box_lengths[1]:.2f} x {box_lengths[2]:.2f} nm "
                f"toward production-start density {PRODUCTION_START_DENSITY_G_CM3:.2f} g/cm^3"
            )
            openmm_memory_preflight(interchange.topology.n_atoms, f"{built.system_name}: periodic NPT OpenMM context", OPENMM_PLATFORM_NAME)
            periodic_integrator = LangevinMiddleIntegrator(periodic_temperature, friction, periodic_time_step)
            periodic_simulation = interchange.to_openmm_simulation(
                integrator=periodic_integrator,
                combine_nonbonded_forces=False,
                additional_forces=[MonteCarloBarostat(pressure, periodic_temperature, 25)],
                **openmm_platform_kwargs(),
            )
            print(f"{built.system_name}: OpenMM platform = {periodic_simulation.context.getPlatform().getName()}")
            nonbonded_cutoff_nm = maximum_nonbonded_cutoff_nm(periodic_simulation.system)
            minimum_box_length_nm = MIN_PERIODIC_BOX_CUTOFF_MULTIPLIER * nonbonded_cutoff_nm
            print(
                f"{built.system_name}: cutoff guard stops NPT before the minimum box length "
                f"falls below {minimum_box_length_nm:.2f} nm "
                f"({MIN_PERIODIC_BOX_CUTOFF_MULTIPLIER:.1f} x {nonbonded_cutoff_nm:.2f} nm cutoff)"
            )
            print(f"{built.system_name}: running periodic minimization before NPT dynamics")
            periodic_simulation.minimizeEnergy(maxIterations=OPENMM_MINIMIZATION_MAX_ITERATIONS)
            periodic_dcd_path = openmm_dir / f"{system_name}_periodic_npt_trajectory.dcd"
            periodic_state_data_path = openmm_dir / f"{system_name}_periodic_npt_state_data.csv"
            attach_openmm_reporters(periodic_simulation, periodic_dcd_path, periodic_state_data_path, periodic_interval, include_density=True)

            steps_run = 0
            periodic_elapsed_ns = 0.0
            current_periodic_timestep_fs = PERIODIC_TIMESTEP_FS
            current_density = 0.0
            stopped_by_cutoff_guard = False
            final_stage = MELT_PREPARATION_PROTOCOL[-1]
            final_temperature_k = float(final_stage["temperature_k"])
            final_pressure_atm = float(final_stage["pressure_atm"])
            for stage_idx, stage in enumerate(MELT_PREPARATION_PROTOCOL, start=1):
                stage_name = str(stage["name"])
                target_density = float(stage["target_density_g_cm3"])
                stage_temperature_k = float(stage["temperature_k"])
                stage_pressure_atm = float(stage["pressure_atm"])
                stage_max_steps = simulation_steps(float(stage["max_duration_ns"]), PERIODIC_TIMESTEP_FS)
                stage_steps_run = 0
                set_barostat_conditions(periodic_simulation, stage_temperature_k, stage_pressure_atm)
                print(
                    f"{built.system_name}: stage {stage_idx}/{len(MELT_PREPARATION_PROTOCOL)} '{stage_name}' "
                    f"until density {target_density:.2f} g/cm^3 "
                    f"at {stage_temperature_k:.0f} K, {stage_pressure_atm:.1f} atm"
                )
                while stage_steps_run < stage_max_steps and current_density < target_density:
                    steps_this_chunk = min(periodic_chunk_steps, stage_max_steps - stage_steps_run)
                    current_periodic_timestep_fs = run_recoverable_openmm_chunk(
                        periodic_simulation,
                        steps_this_chunk,
                        current_periodic_timestep_fs,
                        f"{built.system_name}: periodic NPT stage '{stage_name}'",
                    )
                    steps_run += steps_this_chunk
                    stage_steps_run += steps_this_chunk
                    periodic_elapsed_ns += steps_this_chunk * current_periodic_timestep_fs / 1_000_000.0
                    state = periodic_simulation.context.getState(getPositions=True)
                    box_vectors_nm = state.getPeriodicBoxVectors(asNumpy=True).value_in_unit(omm_unit.nanometer)
                    lengths = box_lengths_nm(box_vectors_nm)
                    if np.min(lengths) <= minimum_box_length_nm:
                        stopped_by_cutoff_guard = True
                        print(
                            f"{built.system_name}: stopping NPT because the minimum box length "
                            f"{np.min(lengths):.2f} nm reached the cutoff guard {minimum_box_length_nm:.2f} nm"
                        )
                        break
                    current_density = density_from_box_g_cm3(system_mass_da, box_vectors_nm)
                    print(
                        f"{built.system_name}: NPT {periodic_elapsed_ns:.4f} ns, stage='{stage_name}', "
                        f"dt={current_periodic_timestep_fs:.4f} fs, "
                        f"density={current_density:.3f} g/cm^3, "
                        f"box={lengths[0]:.2f} x {lengths[1]:.2f} x {lengths[2]:.2f} nm"
                    )
                if stopped_by_cutoff_guard:
                    break
                if current_density >= target_density:
                    print(f"{built.system_name}: reached {stage_name} density threshold ({current_density:.3f} g/cm^3)")
                else:
                    print(
                        f"{built.system_name}: stage '{stage_name}' hit its safety cap at "
                        f"{current_density:.3f} g/cm^3; moving to the next protocol stage"
                    )
                if current_density >= PRODUCTION_START_DENSITY_G_CM3:
                    print(f"{built.system_name}: production-start density threshold reached")
                    break

            print(f"{built.system_name}: periodic trajectory saved to {periodic_dcd_path.relative_to(EXAMPLES_ROOT)}")
            print(f"{built.system_name}: periodic state data saved to {periodic_state_data_path.relative_to(EXAMPLES_ROOT)}")
            if current_density < PRODUCTION_START_DENSITY_G_CM3:
                print(
                    f"{built.system_name}: warning: final density {current_density:.3f} g/cm^3 "
                    f"is below production-start target {PRODUCTION_START_DENSITY_G_CM3:.3f} g/cm^3"
                )
            periodic_state = periodic_simulation.context.getState(getEnergy=True, getPositions=True, getVelocities=True)
            periodic_state_path = openmm_dir / f"{system_name}_periodic_npt_state.xml"
            periodic_state_path.write_text(XmlSerializer.serialize(periodic_state))
            production_files = write_production_start_files(
                periodic_simulation,
                openff_molecules_by_system[built.system_name],
                openmm_dir,
                system_name,
                system_mass_da,
                steps_run,
                PERIODIC_TIMESTEP_FS,
                final_temperature_k,
                final_pressure_atm,
                PRODUCTION_START_DENSITY_G_CM3,
            )
            print(f"{built.system_name}: periodic NPT potential energy: {periodic_state.getPotentialEnergy()}")
            print(f"{built.system_name}: production-start files:")
            for output_path in production_files.values():
                print(f"  {output_path.relative_to(EXAMPLES_ROOT)}")
else:
    print("OpenMM run skipped because no Interchange was created or RUN_VACUUM_COLLAPSE is False.")


m5: creating non-periodic vacuum OpenMM simulation
m5: running vacuum minimization


## 10. Notes for Production Runs

For the production target, set `USE_PRODUCTION_SIZE = True` and run one system at a time. The expected sodium counts are 9,600 for `m3`, 5,600 for `m5`, and 4,000 for `m7`. The vacuum/NPT protocol is intended to generate dense, physically relaxed starting coordinates for later production MD rather than to be the final scientific trajectory.

The melt-preparation protocol switches automatically by density: aggressive high-temperature/high-pressure NPT is used only while the system is very dilute, then the protocol relaxes toward the target production-start density. Duration settings are safety caps; density thresholds determine normal stage transitions.

Initial chain placement also self-tunes: if the generated coordinates fail the nearest-neighbor distance check, the build cell increases the chain-start grid spacing and rebuilds the current system automatically.

When `RUN_PERIODIC_NPT = True`, the notebook writes a `production_start/` directory containing a boxed PDB, OpenMM `State` XML, binary checkpoint, compressed NumPy arrays for positions and box vectors, and a JSON manifest with the final density. Use those files to start subsequent simulations at the desired temperature, pressure, and ensemble.
